<a href="https://colab.research.google.com/github/Pankaj-Kar/blank-app/blob/main/Final_PL_Composite_Indicator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Reliability**

In [ ]:
# ============================================================
# COMBINED IEEE / BERC BASED RELIABILITY SCORE CALCULATION
# Feeder-wise + Year-wise + Overall Reliability
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files
import io

# ============================================================
# 1. UPLOAD FILE
# ============================================================

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

if file_name.endswith(".csv"):
    df = pd.read_csv(io.BytesIO(uploaded[file_name]))
else:
    df = pd.read_excel(io.BytesIO(uploaded[file_name]))

df.columns = df.columns.astype(str).str.strip()

print("File loaded successfully:", file_name)
print("Shape:", df.shape)
display(df.head())

# ============================================================
# 2. COLUMN CONFIGURATION
# ============================================================

date_col = "Complaint Date"
feeder_col = "Feeder Name"
duration_col = "Interruption Duration (min)"
consumer_col = "Affected Consumers"
load_col = "Load Affected (kW)"
severity_col = "Severity Level"
fault_col = "Fault Type"
reason_col = "Reason Category"
equipment_col = "Equipment ID"

# Fix possible column spacing issue
if "Affected Consumers " in df.columns:
    df.rename(columns={"Affected Consumers ": "Affected Consumers"}, inplace=True)

# ============================================================
# 3. DATA CLEANING
# ============================================================

required_cols = [feeder_col, duration_col, consumer_col]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

for col in [duration_col, consumer_col, load_col]:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

if date_col in df.columns:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df["Year"] = df[date_col].dt.year
else:
    df["Year"] = "Unknown"

if severity_col not in df.columns:
    df[severity_col] = "Low"

if fault_col not in df.columns:
    df[fault_col] = "Unknown"

if reason_col not in df.columns:
    df[reason_col] = "Unknown"

if equipment_col not in df.columns:
    df[equipment_col] = "Unknown"

df[feeder_col] = df[feeder_col].fillna("Unknown Feeder")
df[reason_col] = df[reason_col].fillna("Unknown")
df[equipment_col] = df[equipment_col].fillna("Unknown")

df = df[df[consumer_col] > 0].copy()

# ============================================================
# 4. SEVERITY AND FAULT CRITICALITY MAPPING
# ============================================================

severity_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Critical": 4
}

fault_map = {
    "Grid Failure": 5,
    "Transformer Fault": 4,
    "Feeder Trip": 3,
    "Line Fault": 3,
    "Cable Fault": 3,
    "Overload": 3,
    "Voltage Issue": 2,
    "Fuse Fault": 2,
    "Weather Fault": 2,
    "Lightning": 2,
    "Storm": 2,
    "Meter Fault": 1,
    "Connection Fault": 1,
    "Unknown": 1
}

df["Severity Score"] = df[severity_col].map(severity_map).fillna(1)
df["Fault Criticality"] = df[fault_col].map(fault_map).fillna(1)

# ============================================================
# 5. IEEE / BERC THRESHOLD NORMALIZATION FUNCTIONS
# ============================================================

def clamp_0_1(x):
    return np.clip(x, 0, 1)

def cost_normalize(series, threshold):
    """
    Cost-type indicator:
    Lower value is better.
    IEEE/BERC threshold form:
    N = 1 - X / X_threshold
    """
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    return clamp_0_1(1 - series / threshold)

def benefit_normalize(series, threshold):
    """
    Benefit-type indicator:
    Higher value is better.
    N = X / X_threshold
    """
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    return clamp_0_1(series / threshold)

# ============================================================
# 6. IEEE / BERC PRACTICAL THRESHOLDS
# Modify according to your selected BERC / BREB / IEEE benchmark
# ============================================================

THRESHOLDS = {
    "SAIFI": 10.0,        # interruptions/customer/year
    "SAIDI": 600.0,       # minutes/customer/year
    "CAIDI": 120.0,       # minutes/interruption
    "MAIFI": 5.0,         # momentary interruptions/customer/year
    "LAMBDA": 1.0,        # failures/day
    "LAMBDA_E": 1.0,      # equipment failures/day
    "P_E": 1.0,           # probability limit
    "AVAILABILITY": 100.0,
    "CONSUMER": 100.0,
    "LOAD": 100.0,
    "SEVERITY": 100.0,
    "FAULT": 100.0
}

# ============================================================
# 7. RELIABILITY CLASSIFICATION
# ============================================================

def classify_reliability(score):
    if score >= 90:
        return "Excellent Reliability"
    elif score >= 80:
        return "Very Good Reliability"
    elif score >= 70:
        return "Good Reliability"
    elif score >= 60:
        return "Moderate Reliability"
    elif score >= 50:
        return "Weak Reliability"
    else:
        return "Critical Reliability"

# ============================================================
# 8. CORE RELIABILITY FUNCTION
# ============================================================

def calculate_reliability(data, group_cols):

    results = []

    for keys, group in data.groupby(group_cols):

        if not isinstance(keys, tuple):
            keys = (keys,)

        group = group.copy()

        if date_col in group.columns and group[date_col].notna().any():
            start_date = group[date_col].min()
            end_date = group[date_col].max()
            observation_days = max((end_date - start_date).days + 1, 1)
        else:
            observation_days = 365

        T_minutes = observation_days * 24 * 60

        total_failures = len(group)
        total_duration = group[duration_col].sum()

        max_consumers = group[consumer_col].max()
        max_load = group[load_col].max()

        max_consumers = max_consumers if max_consumers > 0 else 1
        max_load = max_load if max_load > 0 else 1

        # Momentary vs sustained interruption
        group["Interruption Type"] = np.where(
            group[duration_col] < 5,
            "Momentary",
            "Sustained"
        )

        momentary_group = group[group["Interruption Type"] == "Momentary"]
        sustained_group = group[group["Interruption Type"] == "Sustained"]

        momentary_customer_interruptions = momentary_group[consumer_col].sum()
        sustained_customer_interruptions = sustained_group[consumer_col].sum()

        sustained_customer_duration = (
            sustained_group[duration_col] *
            sustained_group[consumer_col]
        ).sum()

        # ====================================================
        # IEEE Reliability Indices
        # ====================================================

        lambda_failure = total_failures / observation_days
        lambda_equipment = group[equipment_col].count() / observation_days

        MAIFI = momentary_customer_interruptions / max_consumers
        SAIFI = sustained_customer_interruptions / max_consumers
        SAIDI = sustained_customer_duration / max_consumers
        CAIDI = SAIDI / SAIFI if SAIFI > 0 else 0

        # Dominant cause probability
        dominant_cause = group[reason_col].mode()
        dominant_cause = dominant_cause.iloc[0] if len(dominant_cause) > 0 else "Unknown"

        dominant_cause_count = (group[reason_col] == dominant_cause).sum()
        p_e = dominant_cause_count / total_failures if total_failures > 0 else 0

        # ====================================================
        # Direct Weighted Reliability Scores
        # ====================================================

        availability_score = (1 - total_duration / T_minutes) * 100

        consumer_score = (
            1 -
            (group[duration_col] * group[consumer_col]).sum()
            / (T_minutes * max_consumers)
        ) * 100

        load_score = (
            1 -
            (group[duration_col] * group[load_col]).sum()
            / (T_minutes * max_load)
        ) * 100

        severity_score = (
            1 -
            (group[duration_col] * group["Severity Score"]).sum()
            / (T_minutes * group["Severity Score"].max())
        ) * 100

        fault_score = (
            1 -
            (group[duration_col] * group["Fault Criticality"]).sum()
            / (T_minutes * group["Fault Criticality"].max())
        ) * 100

        availability_score = np.clip(availability_score, 0, 100)
        consumer_score = np.clip(consumer_score, 0, 100)
        load_score = np.clip(load_score, 0, 100)
        severity_score = np.clip(severity_score, 0, 100)
        fault_score = np.clip(fault_score, 0, 100)

        results.append({
            **dict(zip(group_cols, keys)),

            "Observation Days": observation_days,
            "Total Failures": total_failures,
            "Total Outage Duration (min)": total_duration,

            "lambda": lambda_failure,
            "lambda_e": lambda_equipment,
            "p_e": p_e,

            "MAIFI": MAIFI,
            "SAIFI": SAIFI,
            "SAIDI_min": SAIDI,
            "SAIDI_hr": SAIDI / 60,
            "CAIDI_min": CAIDI,
            "CAIDI_hr": CAIDI / 60,

            "Availability Reliability (%)": availability_score,
            "Consumer Weighted Reliability (%)": consumer_score,
            "Load Weighted Reliability (%)": load_score,
            "Severity Weighted Reliability (%)": severity_score,
            "Fault Criticality Reliability (%)": fault_score,

            "Dominant Cause": dominant_cause
        })

    result = pd.DataFrame(results)

    # ========================================================
    # IEEE / BERC NORMALIZATION
    # ========================================================

    result["N_lambda"] = cost_normalize(result["lambda"], THRESHOLDS["LAMBDA"])
    result["N_lambda_e"] = cost_normalize(result["lambda_e"], THRESHOLDS["LAMBDA_E"])
    result["N_p_e"] = cost_normalize(result["p_e"], THRESHOLDS["P_E"])

    result["N_MAIFI"] = cost_normalize(result["MAIFI"], THRESHOLDS["MAIFI"])
    result["N_SAIFI"] = cost_normalize(result["SAIFI"], THRESHOLDS["SAIFI"])
    result["N_SAIDI"] = cost_normalize(result["SAIDI_min"], THRESHOLDS["SAIDI"])
    result["N_CAIDI"] = cost_normalize(result["CAIDI_min"], THRESHOLDS["CAIDI"])

    result["N_Availability"] = benefit_normalize(
        result["Availability Reliability (%)"],
        THRESHOLDS["AVAILABILITY"]
    )

    result["N_Consumer"] = benefit_normalize(
        result["Consumer Weighted Reliability (%)"],
        THRESHOLDS["CONSUMER"]
    )

    result["N_Load"] = benefit_normalize(
        result["Load Weighted Reliability (%)"],
        THRESHOLDS["LOAD"]
    )

    result["N_Severity"] = benefit_normalize(
        result["Severity Weighted Reliability (%)"],
        THRESHOLDS["SEVERITY"]
    )

    result["N_Fault"] = benefit_normalize(
        result["Fault Criticality Reliability (%)"],
        THRESHOLDS["FAULT"]
    )

    # ========================================================
    # FINAL COMPOSITE RELIABILITY SCORE
    # ========================================================

    weights = {
        "N_SAIDI": 0.20,
        "N_SAIFI": 0.18,
        "N_CAIDI": 0.10,
        "N_MAIFI": 0.07,
        "N_lambda": 0.08,
        "N_lambda_e": 0.07,
        "N_p_e": 0.05,
        "N_Consumer": 0.10,
        "N_Load": 0.05,
        "N_Severity": 0.05,
        "N_Fault": 0.05
    }

    result["Reliability_Index"] = sum(
        result[col] * weight for col, weight in weights.items()
    )

    result["Reliability_Percent"] = result["Reliability_Index"] * 100

    result["Reliability_Class"] = result["Reliability_Percent"].apply(
        classify_reliability
    )

    return result.sort_values("Reliability_Percent", ascending=False)

# ============================================================
# 9. FEEDER-WISE RELIABILITY
# ============================================================

feeder_reliability = calculate_reliability(
    df,
    group_cols=[feeder_col]
)

print("\nFEEDER-WISE RELIABILITY RESULT")
display(feeder_reliability.round(3))

# ============================================================
# 10. YEAR-WISE RELIABILITY
# ============================================================

year_reliability = calculate_reliability(
    df,
    group_cols=["Year"]
)

print("\nYEAR-WISE RELIABILITY RESULT")
display(year_reliability.round(3))

# ============================================================
# 11. FEEDER-WISE AND YEAR-WISE RELIABILITY
# ============================================================

feeder_year_reliability = calculate_reliability(
    df,
    group_cols=[feeder_col, "Year"]
)

print("\nFEEDER-WISE AND YEAR-WISE RELIABILITY RESULT")
display(feeder_year_reliability.round(3))

# ============================================================
# 12. FINAL VALUE FOR COMPOSITE R-A-S-CE CODE
# ============================================================

reliability_value = feeder_year_reliability[
    [feeder_col, "Year", "Reliability_Percent"]
].copy()

print("\nRELIABILITY VALUE FOR FINAL COMPOSITE CODE")
display(reliability_value.round(2))

# ============================================================
# 13. EXPORT RESULTS
# ============================================================

output_file = "Combined_IEEE_BERC_Reliability_Result.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    feeder_reliability.to_excel(writer, sheet_name="Feeder Reliability", index=False)
    year_reliability.to_excel(writer, sheet_name="Year Reliability", index=False)
    feeder_year_reliability.to_excel(writer, sheet_name="Feeder Year Reliability", index=False)
    reliability_value.to_excel(writer, sheet_name="Reliability Value", index=False)

print("\nReliability calculation completed successfully.")
print("Output file:", output_file)

# Optional download
# files.download(output_file)

Saving Updated 5000 Complaints Sreepur.xlsx to Updated 5000 Complaints Sreepur (15).xlsx
File loaded successfully: Updated 5000 Complaints Sreepur (15).xlsx
Shape: (5000, 36)


,Complaint ID,Ticket Number,Complainant Name,Phone Number,Complaintn Receiver,Complaint Date,Complaint Time,Shutdown Date,Shutdown Time,Restart Date,...,Affected Consumers,Load Affected (kW),Affected Area.1,Reason of Interruption,Reason Category,Reason of Interruption.1,Fault Category,Interruption Description,Action Taken,Fault Type
0,C05952,TKT-05952,সোহেল খান,1563963655,রফিক (EMP412),01-Jan-2022,03:02,01-Jan-2022,03:06,01-Jan-2022,...,305,1716,কাজী তলা বাজার,ট্রান্সফরমার বিকল হওয়ায় সম্পূর্ণ এলাকায় বিদ...,Transformer Fault,Transformer Failure,Transformer Fault,ট্রান্সফরমার বিকল হওয়ায় সম্পূর্ণ এলাকায় বিদ...,ত্রুটি সংশোধন করে পুনঃসংযোগ দেওয়া হয়েছে,Transformer Fault
1,C00522,TKT-00522,মমিন বেপারী,1273201828,হাসান (EMP225),01-Jan-2022,03:30,01-Jan-2022,03:37,01-Jan-2022,...,216,3183,মুলাইদ ব্লক,ট্রান্সফরমার বিকল হওয়ায় সম্পূর্ণ এলাকায় বিদ...,Transformer Fault,Transformer Failure,Transformer Fault,ট্রান্সফরমার বিকল হওয়ায় সম্পূর্ণ এলাকায় বিদ...,লাইন মেরামত করে চালু করা হয়েছে,Transformer Fault
2,C00669,TKT-00669,সালাম মল্লিক,1194051674,শরিফ (EMP318),01-Jan-2022,10:30,01-Jan-2022,10:40,01-Jan-2022,...,342,3487,"রংগেলা বাজার সংলগ্ন এলাকা, তার ছিঁড়ে গেছে",বজ্রপাতের কারণে ফিডার ট্রিপ করে সরবরাহ বন্ধ হয়,Weather Fault,Lightning Fault,Weather Fault,বজ্রপাতের কারণে ফিডার ট্রিপ করে সরবরাহ বন্ধ হয়,লাইন মেরামত করে চালু করা হয়েছে,Weather Fault
3,C04133,TKT-04133,আব্দুল শেখ,1024619306,নাঈম (EMP390),01-Jan-2022,14:53,01-Jan-2022,14:51,01-Jan-2022,...,609,3382,বৈরাইদের চালা পাড়া,লোড অসমতার কারণে ভোল্টেজ স্বাভাবিকের নিচে নেমে...,Voltage Issue,Voltage Drop,Voltage Issue,লোড অসমতার কারণে ভোল্টেজ স্বাভাবিকের নিচে নেমে...,লোড ব্যালেন্স করা হয়েছে,Voltage Issue
4,C05386,TKT-05386,হাসান খান,1994956873,নাঈম (EMP390),01-Jan-2022,16:04,01-Jan-2022,16:11,01-Jan-2022,...,380,1810,"মাওনা চৌরাস্তা সংলগ্ন এলাকা, মাওনা বাসস্ট্যান্...",বজ্রপাতের কারণে ফিডার ট্রিপ করে সরবরাহ বন্ধ হয়,Weather Fault,Lightning Fault,Weather Fault,বজ্রপাতের কারণে ফিডার ট্রিপ করে সরবরাহ বন্ধ হয়,যন্ত্রাংশ পরিবর্তন করা হয়েছে,Weather Fault



FEEDER-WISE RELIABILITY RESULT


,Feeder Name,Observation Days,Total Failures,Total Outage Duration (min),lambda,lambda_e,p_e,MAIFI,SAIFI,SAIDI_min,...,N_SAIDI,N_CAIDI,N_Availability,N_Consumer,N_Load,N_Severity,N_Fault,Reliability_Index,Reliability_Percent,Reliability_Class
8,Feeder-7,1448,431,47369,0.298,0.298,0.239,0.0,273.444,29515.576,...,0.0,0.101,0.977,0.986,0.986,0.983,0.988,0.470,46.986,Critical Reliability
10,Feeder-9,1458,441,52811,0.302,0.302,0.234,0.0,275.289,32633.591,...,0.0,0.012,0.975,0.984,0.985,0.981,0.986,0.460,46.019,Critical Reliability
2,Feeder-11,1461,454,53956,0.311,0.311,0.238,0.0,285.374,33532.382,...,0.0,0.021,0.974,0.984,0.985,0.981,0.986,0.460,45.957,Critical Reliability
6,Feeder-5,1461,444,53912,0.304,0.304,0.225,0.0,272.023,32596.444,...,0.0,0.001,0.974,0.985,0.983,0.981,0.986,0.459,45.926,Critical Reliability
7,Feeder-6,1440,440,55107,0.306,0.306,0.234,0.0,282.602,35324.487,...,0.0,0.000,0.973,0.983,0.983,0.980,0.985,0.458,45.819,Critical Reliability
9,Feeder-8,1459,445,58321,0.305,0.305,0.238,0.0,281.623,36690.706,...,0.0,0.000,0.972,0.983,0.983,0.980,0.985,0.458,45.797,Critical Reliability
3,Feeder-2,1455,454,57927,0.312,0.312,0.244,0.0,278.694,35858.289,...,0.0,0.000,0.972,0.983,0.983,0.980,0.985,0.457,45.664,Critical Reliability
1,Feeder-10,1459,469,59956,0.321,0.321,0.228,0.0,290.714,37144.514,...,0.0,0.000,0.971,0.982,0.982,0.979,0.984,0.456,45.584,Critical Reliability
0,Feeder-1,1457,473,57747,0.325,0.325,0.230,0.0,294.129,35240.432,...,0.0,0.002,0.972,0.983,0.983,0.980,0.984,0.456,45.559,Critical Reliability
4,Feeder-3,1458,467,58464,0.320,0.320,0.251,0.0,291.564,36712.104,...,0.0,0.000,0.972,0.983,0.983,0.979,0.985,0.455,45.498,Critical Reliability



YEAR-WISE RELIABILITY RESULT


...


FEEDER-WISE AND YEAR-WISE RELIABILITY RESULT


,Feeder Name,Year,Observation Days,Total Failures,Total Outage Duration (min),lambda,lambda_e,p_e,MAIFI,SAIFI,...,N_SAIDI,N_CAIDI,N_Availability,N_Consumer,N_Load,N_Severity,N_Fault,Reliability_Index,Reliability_Percent,Reliability_Class
34,Feeder-7,2024,359,107,10671,0.298,0.298,0.243,0.0,72.427,...,0.0,0.187,0.979,0.986,0.987,0.985,0.989,0.478,47.850,Critical Reliability
9,Feeder-11,2023,351,100,10703,0.285,0.285,0.230,0.0,60.035,...,0.0,0.151,0.979,0.988,0.988,0.984,0.989,0.478,47.771,Critical Reliability
33,Feeder-7,2023,361,106,10996,0.294,0.294,0.226,0.0,65.073,...,0.0,0.151,0.979,0.987,0.986,0.984,0.989,0.476,47.642,Critical Reliability
42,Feeder-9,2024,359,109,12211,0.304,0.304,0.229,0.0,67.576,...,0.0,0.117,0.976,0.986,0.986,0.982,0.988,0.471,47.103,Critical Reliability
25,Feeder-5,2023,363,104,11703,0.287,0.287,0.250,0.0,63.859,...,0.0,0.091,0.978,0.987,0.985,0.983,0.988,0.470,47.008,Critical Reliability
14,Feeder-2,2024,364,110,11997,0.302,0.302,0.273,0.0,67.262,...,0.0,0.114,0.977,0.986,0.986,0.983,0.987,0.469,46.892,Critical Reliability
24,Feeder-5,2022,360,94,11193,0.261,0.261,0.223,0.0,55.792,...,0.0,0.000,0.978,0.987,0.987,0.983,0.989,0.466,46.630,Critical Reliability
38,Feeder-8,2024,364,109,12466,0.299,0.299,0.275,0.0,67.882,...,0.0,0.052,0.976,0.985,0.985,0.982,0.989,0.463,46.291,Critical Reliability
32,Feeder-7,2022,353,110,12763,0.312,0.312,0.245,0.0,69.026,...,0.0,0.055,0.975,0.985,0.984,0.981,0.987,0.463,46.263,Critical Reliability
7,Feeder-10,2025,360,101,13459,0.281,0.281,0.228,0.0,60.861,...,0.0,0.000,0.974,0.985,0.984,0.980,0.985,0.462,46.247,Critical Reliability



RELIABILITY VALUE FOR FINAL COMPOSITE CODE


,Feeder Name,Year,Reliability_Percent
34,Feeder-7,2024,47.85
9,Feeder-11,2023,47.77
33,Feeder-7,2023,47.64
42,Feeder-9,2024,47.10
25,Feeder-5,2023,47.01
14,Feeder-2,2024,46.89
24,Feeder-5,2022,46.63
38,Feeder-8,2024,46.29
32,Feeder-7,2022,46.26
7,Feeder-10,2025,46.25



Reliability calculation completed successfully.
Output file: Combined_IEEE_BERC_Reliability_Result.xlsx


# **Availability**

In [ ]:
# ============================================================
# YEAR-WISE + FEEDER-YEAR-WISE COMPOSITE AVAILABILITY ANALYSIS
# IEEE / BERC Threshold-Based Normalization
# Google Colab Compatible
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files

# ============================================================
# 1. UPLOAD AND LOAD DATASET
# ============================================================

print("Upload your Excel/CSV file:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

if file_name.lower().endswith(".csv"):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name)

df.columns = df.columns.astype(str).str.strip()
df = df.loc[:, ~df.columns.duplicated()]

print("Loaded:", file_name)
print("Shape:", df.shape)
display(df.head())

# ============================================================
# 2. SYSTEM CONSTANTS
# ============================================================

NT = 5000        # Total customers
LT = 5000        # Total system load in kW
T_YEAR = 8760    # Annual observation time in hours

# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def safe_divide(a, b):
    if b == 0 or pd.isna(b):
        return np.nan
    return a / b

def clamp_value(x, low=0.0, high=1.0):
    if pd.isna(x):
        return np.nan
    return max(low, min(high, x))

def threshold_cost_norm(x, threshold):
    if pd.isna(x) or pd.isna(threshold) or threshold == 0:
        return np.nan
    return clamp_value(1 - x / threshold)

def threshold_benefit_norm(x, threshold):
    if pd.isna(x) or pd.isna(threshold) or threshold == 0:
        return np.nan
    return clamp_value(x / threshold)

def weighted_average(row, weights):
    total_weight = 0
    score = 0

    for col, w in weights.items():
        if col in row.index and pd.notna(row[col]):
            score += row[col] * w
            total_weight += w

    if total_weight == 0:
        return np.nan

    return score / total_weight

# ============================================================
# 4. DATE-TIME PROCESSING
# ============================================================

required_dt_cols = [
    "Shutdown Date", "Shutdown Time",
    "Restart Date", "Restart Time",
    "Complaint Date", "Complaint Time"
]

for col in required_dt_cols:
    if col not in df.columns:
        df[col] = np.nan

df["shutdown_dt"] = pd.to_datetime(
    df["Shutdown Date"].astype(str) + " " + df["Shutdown Time"].astype(str),
    errors="coerce"
)

df["restart_dt"] = pd.to_datetime(
    df["Restart Date"].astype(str) + " " + df["Restart Time"].astype(str),
    errors="coerce"
)

df["complaint_dt"] = pd.to_datetime(
    df["Complaint Date"].astype(str) + " " + df["Complaint Time"].astype(str),
    errors="coerce"
)

# ============================================================
# 5. YEAR COLUMN
# ============================================================

df["Year"] = df["complaint_dt"].dt.year

df["Year"] = df["Year"].fillna(
    df["shutdown_dt"].dt.year
)

df["Year"] = df["Year"].fillna(
    df["restart_dt"].dt.year
)

df = df.dropna(subset=["Year"])
df["Year"] = df["Year"].astype(int)

# ============================================================
# 6. NUMERIC COLUMN PROCESSING
# ============================================================

numeric_cols = [
    "Interruption Duration (min)",
    "Resolve Duration",
    "Affected Consumers",
    "Load Affected (kW)"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = safe_numeric(df[col])
    else:
        df[col] = 0

# ============================================================
# 7. DURATION CALCULATION
# ============================================================

df["Duration_min"] = (
    df["restart_dt"] - df["shutdown_dt"]
).dt.total_seconds() / 60

df["restore_min"] = (
    df["restart_dt"] - df["complaint_dt"]
).dt.total_seconds() / 60

df["Duration_min"] = df["Duration_min"].fillna(
    df["Interruption Duration (min)"]
)

df["restore_min"] = df["restore_min"].fillna(
    df["Resolve Duration"]
)

df["Duration_min"] = df["Duration_min"].clip(lower=0)
df["restore_min"] = df["restore_min"].clip(lower=0)

df["Duration_hr"] = df["Duration_min"] / 60
df["restore_hr"] = df["restore_min"] / 60

# ============================================================
# 8. FEEDER COLUMN HANDLING
# ============================================================

if "Feeder Name" not in df.columns:
    raise ValueError("Feeder Name column is required.")

df["Feeder Name"] = df["Feeder Name"].fillna("Unknown Feeder")

# ============================================================
# 9. IEEE / BERC THRESHOLDS
# ============================================================

AVAILABILITY_THRESHOLDS = {
    "D_MAX": 8760,
    "U_MAX": 8760,
    "MTTR_MAX": 24,
    "MTRS_MAX": 24,
    "MTBF_MAX": 8760,
    "Ao_MAX": 1.0,
    "CAI_MAX": 1.0,
    "LAI_MAX": 1.0,
    "ENS_MAX": LT * T_YEAR
}

availability_weights = {
    "N_A": 0.15,
    "N_U": 0.10,
    "N_D": 0.10,
    "N_MTTR": 0.10,
    "N_MTRS": 0.10,
    "N_MTBF": 0.10,
    "N_Ao": 0.10,
    "N_CAI": 0.10,
    "N_LAI": 0.10,
    "N_ENS": 0.05
}

# ============================================================
# 10. AVAILABILITY CALCULATION FUNCTION
# ============================================================

def availability_metrics(group):

    group = group.copy()
    N = len(group)

    valid_time = group.dropna(subset=["shutdown_dt", "restart_dt"])

    if len(valid_time) > 0:
        t_start = valid_time["shutdown_dt"].min()
        t_end = valid_time["restart_dt"].max()
        T_obs = (t_end - t_start).total_seconds() / 3600
    else:
        T_obs = T_YEAR

    if pd.isna(T_obs) or T_obs <= 0:
        T_obs = T_YEAR

    D = group["Duration_hr"].sum(skipna=True)
    U = max(T_obs - D, 0)

    total_repair_time = group["Duration_hr"].sum(skipna=True)
    total_restore_time = group["restore_hr"].sum(skipna=True)

    MTTR = safe_divide(total_repair_time, N)
    MTRS = safe_divide(total_restore_time, N)
    MTBF = safe_divide(U, N)

    Ao = (
        MTBF / (MTBF + MTTR)
        if pd.notna(MTBF) and pd.notna(MTTR) and (MTBF + MTTR) > 0
        else np.nan
    )

    customer_outage_time = (
        group["Duration_hr"].fillna(0) *
        group["Affected Consumers"].fillna(0)
    ).sum()

    customer_demand_time = NT * T_obs

    CAI = (
        1 - customer_outage_time / customer_demand_time
        if customer_demand_time > 0
        else np.nan
    )

    ENS = (
        group["Load Affected (kW)"].fillna(0) *
        group["Duration_hr"].fillna(0)
    ).sum()

    E_total = LT * T_obs

    LAI = (
        1 - ENS / E_total
        if E_total > 0
        else np.nan
    )

    A = 1 - D / T_obs if T_obs > 0 else np.nan
    ASAI = safe_divide(U, T_obs)

    return pd.Series({
        "Number of Failures": N,
        "Observation Time T (hr)": T_obs,
        "Total Outage Duration D (hr)": D,
        "Uptime U (hr)": U,
        "Availability A = 1-D/T": A,
        "ASAI": ASAI,
        "Total Repair Time (hr)": total_repair_time,
        "Total Restore Time (hr)": total_restore_time,
        "MTTR (hr)": MTTR,
        "MTRS (hr)": MTRS,
        "MTBF (hr)": MTBF,
        "Operational Availability Ao": Ao,
        "Customer Outage Time (customer-hr)": customer_outage_time,
        "Customer Demand Time (customer-hr)": customer_demand_time,
        "CAI": CAI,
        "ENS (kWh)": ENS,
        "Total Energy Demand E_total (kWh)": E_total,
        "LAI": LAI
    })

# ============================================================
# 11. NORMALIZATION FUNCTION
# ============================================================

def apply_availability_normalization(result):

    result["N_A"] = result.apply(
        lambda r: threshold_cost_norm(
            r["Total Outage Duration D (hr)"],
            r["Observation Time T (hr)"]
        ),
        axis=1
    )

    result["N_U"] = result.apply(
        lambda r: threshold_benefit_norm(
            r["Uptime U (hr)"],
            r["Observation Time T (hr)"]
        ),
        axis=1
    )

    result["N_D"] = result["Total Outage Duration D (hr)"].apply(
        lambda x: threshold_cost_norm(x, AVAILABILITY_THRESHOLDS["D_MAX"])
    )

    result["N_MTTR"] = result["MTTR (hr)"].apply(
        lambda x: threshold_cost_norm(x, AVAILABILITY_THRESHOLDS["MTTR_MAX"])
    )

    result["N_MTRS"] = result["MTRS (hr)"].apply(
        lambda x: threshold_cost_norm(x, AVAILABILITY_THRESHOLDS["MTRS_MAX"])
    )

    result["N_MTBF"] = result["MTBF (hr)"].apply(
        lambda x: threshold_benefit_norm(x, AVAILABILITY_THRESHOLDS["MTBF_MAX"])
    )

    result["N_Ao"] = result["Operational Availability Ao"].apply(
        lambda x: threshold_benefit_norm(x, AVAILABILITY_THRESHOLDS["Ao_MAX"])
    )

    result["N_CAI"] = result["CAI"].apply(
        lambda x: threshold_benefit_norm(x, AVAILABILITY_THRESHOLDS["CAI_MAX"])
    )

    result["N_LAI"] = result["LAI"].apply(
        lambda x: threshold_benefit_norm(x, AVAILABILITY_THRESHOLDS["LAI_MAX"])
    )

    result["N_ENS"] = result["ENS (kWh)"].apply(
        lambda x: threshold_cost_norm(x, AVAILABILITY_THRESHOLDS["ENS_MAX"])
    )

    result["Composite_Availability_Index"] = result.apply(
        lambda row: weighted_average(row, availability_weights),
        axis=1
    )

    result["Composite_Availability_Percent"] = (
        result["Composite_Availability_Index"] * 100
    )

    return result

# ============================================================
# 12. CLASSIFICATION
# ============================================================

def classify_availability(x):
    if pd.isna(x):
        return "Insufficient Data"
    elif x >= 95:
        return "Excellent Availability"
    elif x >= 90:
        return "High Availability"
    elif x >= 80:
        return "Moderate Availability"
    elif x >= 60:
        return "Low Availability"
    else:
        return "Critical Availability"

# ============================================================
# 13. YEAR-WISE AVAILABILITY
# ============================================================

year_results = (
    df.groupby("Year")
    .apply(availability_metrics)
    .reset_index()
)

year_results = apply_availability_normalization(year_results)

year_results["Availability_Class"] = year_results[
    "Composite_Availability_Percent"
].apply(classify_availability)

print("\nYEAR-WISE AVAILABILITY RESULT")
display(year_results.round(4))

# ============================================================
# 14. FEEDER-WISE AVAILABILITY
# ============================================================

feeder_results = (
    df.groupby("Feeder Name")
    .apply(availability_metrics)
    .reset_index()
)

feeder_results = apply_availability_normalization(feeder_results)

feeder_results["Availability_Class"] = feeder_results[
    "Composite_Availability_Percent"
].apply(classify_availability)

print("\nFEEDER-WISE AVAILABILITY RESULT")
display(feeder_results.round(4))

# ============================================================
# 15. FEEDER-WISE + YEAR-WISE AVAILABILITY
# ============================================================

feeder_year_results = (
    df.groupby(["Feeder Name", "Year"])
    .apply(availability_metrics)
    .reset_index()
)

feeder_year_results = apply_availability_normalization(feeder_year_results)

feeder_year_results["Availability_Class"] = feeder_year_results[
    "Composite_Availability_Percent"
].apply(classify_availability)

print("\nFEEDER-WISE YEAR-WISE AVAILABILITY RESULT")
display(feeder_year_results.round(4))

# ============================================================
# 16. FINAL DISPLAY COLUMNS
# ============================================================

raw_parameter_cols = [
    "Year",
    "Feeder Name",
    "Number of Failures",
    "Observation Time T (hr)",
    "Total Outage Duration D (hr)",
    "Uptime U (hr)",
    "Availability A = 1-D/T",
    "ASAI",
    "MTTR (hr)",
    "MTRS (hr)",
    "MTBF (hr)",
    "Operational Availability Ao",
    "CAI",
    "ENS (kWh)",
    "LAI",
    "Composite_Availability_Index",
    "Composite_Availability_Percent",
    "Availability_Class"
]

normalized_cols = [
    "Year",
    "Feeder Name",
    "N_A",
    "N_U",
    "N_D",
    "N_MTTR",
    "N_MTRS",
    "N_MTBF",
    "N_Ao",
    "N_CAI",
    "N_LAI",
    "N_ENS",
    "Composite_Availability_Index",
    "Composite_Availability_Percent"
]

year_raw_cols = [c for c in raw_parameter_cols if c in year_results.columns]
feeder_raw_cols = [c for c in raw_parameter_cols if c in feeder_results.columns]
feeder_year_raw_cols = [c for c in raw_parameter_cols if c in feeder_year_results.columns]

year_norm_cols = [c for c in normalized_cols if c in year_results.columns]
feeder_norm_cols = [c for c in normalized_cols if c in feeder_results.columns]
feeder_year_norm_cols = [c for c in normalized_cols if c in feeder_year_results.columns]

print("\nYEAR-WISE RAW PARAMETERS")
display(year_results[year_raw_cols].round(4))

print("\nYEAR-WISE NORMALIZED PARAMETERS")
display(year_results[year_norm_cols].round(4))

print("\nFEEDER-YEAR RAW PARAMETERS")
display(feeder_year_results[feeder_year_raw_cols].round(4))

print("\nFEEDER-YEAR NORMALIZED PARAMETERS")
display(feeder_year_results[feeder_year_norm_cols].round(4))

# ============================================================
# 17. SUMMARY
# ============================================================

summary_cols = [
    "Composite_Availability_Index",
    "Composite_Availability_Percent",
    "N_A",
    "N_U",
    "N_D",
    "N_MTTR",
    "N_MTRS",
    "N_MTBF",
    "N_Ao",
    "N_CAI",
    "N_LAI",
    "N_ENS"
]

summary = feeder_year_results[summary_cols].describe().T.round(4)

print("\nCOMPOSITE AVAILABILITY SUMMARY")
display(summary)

# ============================================================
# 18. SAVE AVAILABILITY VALUE FOR FINAL COMPOSITE CODE
# ============================================================

availability_value = feeder_year_results[
    ["Feeder Name", "Year", "Composite_Availability_Percent"]
].copy()

availability_value = availability_value.rename(columns={
    "Composite_Availability_Percent": "Availability_Percent"
})

print("\nAvailability value saved successfully for final composite code.")
display(availability_value.round(2))

# ============================================================
# 19. EXPORT OUTPUT
# ============================================================

output_file = "year_wise_feeder_wise_composite_availability_analysis.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    year_results.to_excel(writer, sheet_name="Year_Availability", index=False)
    feeder_results.to_excel(writer, sheet_name="Feeder_Availability", index=False)
    feeder_year_results.to_excel(writer, sheet_name="Feeder_Year_Availability", index=False)
    year_results[year_norm_cols].to_excel(writer, sheet_name="Year_Normalized", index=False)
    feeder_year_results[feeder_year_norm_cols].to_excel(writer, sheet_name="Feeder_Year_Normalized", index=False)
    availability_value.to_excel(writer, sheet_name="Availability_Value", index=False)
    summary.to_excel(writer, sheet_name="Summary")

print("\nSaved:", output_file)

# Optional download
# files.download(output_file)

Upload your Excel/CSV file:


Saving Updated 5000 Complaints Sreepur.xlsx to Updated 5000 Complaints Sreepur (16).xlsx
Loaded: Updated 5000 Complaints Sreepur (16).xlsx
Shape: (5000, 36)


,Complaint ID,Ticket Number,Complainant Name,Phone Number,Complaintn Receiver,Complaint Date,Complaint Time,Shutdown Date,Shutdown Time,Restart Date,...,Affected Consumers,Load Affected (kW),Affected Area.1,Reason of Interruption,Reason Category,Reason of Interruption.1,Fault Category,Interruption Description,Action Taken,Fault Type
0,C05952,TKT-05952,সোহেল খান,1563963655,রফিক (EMP412),01-Jan-2022,03:02,01-Jan-2022,03:06,01-Jan-2022,...,305,1716,কাজী তলা বাজার,ট্রান্সফরমার বিকল হওয়ায় সম্পূর্ণ এলাকায় বিদ...,Transformer Fault,Transformer Failure,Transformer Fault,ট্রান্সফরমার বিকল হওয়ায় সম্পূর্ণ এলাকায় বিদ...,ত্রুটি সংশোধন করে পুনঃসংযোগ দেওয়া হয়েছে,Transformer Fault
1,C00522,TKT-00522,মমিন বেপারী,1273201828,হাসান (EMP225),01-Jan-2022,03:30,01-Jan-2022,03:37,01-Jan-2022,...,216,3183,মুলাইদ ব্লক,ট্রান্সফরমার বিকল হওয়ায় সম্পূর্ণ এলাকায় বিদ...,Transformer Fault,Transformer Failure,Transformer Fault,ট্রান্সফরমার বিকল হওয়ায় সম্পূর্ণ এলাকায় বিদ...,লাইন মেরামত করে চালু করা হয়েছে,Transformer Fault
2,C00669,TKT-00669,সালাম মল্লিক,1194051674,শরিফ (EMP318),01-Jan-2022,10:30,01-Jan-2022,10:40,01-Jan-2022,...,342,3487,"রংগেলা বাজার সংলগ্ন এলাকা, তার ছিঁড়ে গেছে",বজ্রপাতের কারণে ফিডার ট্রিপ করে সরবরাহ বন্ধ হয়,Weather Fault,Lightning Fault,Weather Fault,বজ্রপাতের কারণে ফিডার ট্রিপ করে সরবরাহ বন্ধ হয়,লাইন মেরামত করে চালু করা হয়েছে,Weather Fault
3,C04133,TKT-04133,আব্দুল শেখ,1024619306,নাঈম (EMP390),01-Jan-2022,14:53,01-Jan-2022,14:51,01-Jan-2022,...,609,3382,বৈরাইদের চালা পাড়া,লোড অসমতার কারণে ভোল্টেজ স্বাভাবিকের নিচে নেমে...,Voltage Issue,Voltage Drop,Voltage Issue,লোড অসমতার কারণে ভোল্টেজ স্বাভাবিকের নিচে নেমে...,লোড ব্যালেন্স করা হয়েছে,Voltage Issue
4,C05386,TKT-05386,হাসান খান,1994956873,নাঈম (EMP390),01-Jan-2022,16:04,01-Jan-2022,16:11,01-Jan-2022,...,380,1810,"মাওনা চৌরাস্তা সংলগ্ন এলাকা, মাওনা বাসস্ট্যান্...",বজ্রপাতের কারণে ফিডার ট্রিপ করে সরবরাহ বন্ধ হয়,Weather Fault,Lightning Fault,Weather Fault,বজ্রপাতের কারণে ফিডার ট্রিপ করে সরবরাহ বন্ধ হয়,যন্ত্রাংশ পরিবর্তন করা হয়েছে,Weather Fault



YEAR-WISE AVAILABILITY RESULT


/tmp/ipykernel_21811/3870790713.py:383: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(availability_metrics)


...


COMPOSITE AVAILABILITY SUMMARY


,count,mean,std,min,25%,50%,75%,max
Composite_Availability_Index,44.0,0.8695,0.0030,0.8648,0.8672,0.8690,0.8722,0.8760
Composite_Availability_Percent,44.0,86.9469,0.3003,86.4792,86.7166,86.9048,87.2200,87.5983
N_A,44.0,0.9730,0.0034,0.9662,0.9701,0.9730,0.9756,0.9793
N_U,44.0,0.9730,0.0034,0.9662,0.9701,0.9730,0.9756,0.9793
N_D,44.0,0.9734,0.0034,0.9670,0.9706,0.9734,0.9759,0.9797
N_MTTR,44.0,0.9147,0.0074,0.8991,0.9095,0.9158,0.9203,0.9307
N_MTRS,44.0,0.9129,0.0074,0.8975,0.9081,0.9141,0.9185,0.9288
N_MTBF,44.0,0.0085,0.0008,0.0070,0.0078,0.0087,0.0091,0.0102
N_Ao,44.0,0.9730,0.0034,0.9662,0.9701,0.9730,0.9756,0.9793
N_CAI,44.0,0.9973,0.0004,0.9965,0.9970,0.9973,0.9976,0.9981



Availability value saved successfully for final composite code.


,Feeder Name,Year,Availability_Percent
0,Feeder-1,2022,86.89
1,Feeder-1,2023,86.94
2,Feeder-1,2024,86.74
3,Feeder-1,2025,87.04
4,Feeder-10,2022,87.09
5,Feeder-10,2023,86.51
6,Feeder-10,2024,86.61
7,Feeder-10,2025,86.87
8,Feeder-11,2022,86.82
9,Feeder-11,2023,87.50



Saved: year_wise_feeder_wise_composite_availability_analysis.xlsx


# **Customer Experience**

In [ ]:
# ============================================================
# YEAR-WISE + FEEDER-YEAR-WISE COMPOSITE CUSTOMER EXPERIENCE INDEX
# Threshold-Based Normalization: IEEE / BERC / Utility Standard
# CE = Σ wi Pi
# Google Colab Compatible
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files
import io

# ============================================================
# 1. Upload and Load Dataset
# ============================================================

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

if file_name.lower().endswith(".csv"):
    df = pd.read_csv(io.BytesIO(uploaded[file_name]))
else:
    df = pd.read_excel(io.BytesIO(uploaded[file_name]))

df.columns = df.columns.astype(str).str.strip()
df = df.loc[:, ~df.columns.duplicated()]

print("File loaded successfully")
print("Total records:", len(df))
display(df.head())

# ============================================================
# 2. Required Column Check
# ============================================================

required_cols = [
    "Feeder Name",
    "Interruption Duration (min)",
    "Affected Consumers",
    "Load Affected (kW)"
]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df["Feeder Name"] = df["Feeder Name"].fillna("Unknown Feeder")

# ============================================================
# 3. Numeric Conversion
# ============================================================

numeric_cols = [
    "Interruption Duration (min)",
    "Resolve Duration",
    "Affected Consumers",
    "Load Affected (kW)"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    else:
        df[col] = 0

# ============================================================
# 4. Datetime Conversion
# ============================================================

def make_datetime(date_col, time_col):
    if date_col in df.columns and time_col in df.columns:
        return pd.to_datetime(
            df[date_col].astype(str) + " " + df[time_col].astype(str),
            errors="coerce"
        )
    return pd.NaT

df["complaint_dt"] = make_datetime("Complaint Date", "Complaint Time")
df["shutdown_dt"] = make_datetime("Shutdown Date", "Shutdown Time")
df["restart_dt"] = make_datetime("Restart Date", "Restart Time")

# ============================================================
# 5. Year Column
# ============================================================

df["Year"] = df["complaint_dt"].dt.year
df["Year"] = df["Year"].fillna(df["shutdown_dt"].dt.year)
df["Year"] = df["Year"].fillna(df["restart_dt"].dt.year)

df = df.dropna(subset=["Year"])
df["Year"] = df["Year"].astype(int)

# ============================================================
# 6. Derived Durations
# ============================================================

df["Detection_Delay_min"] = (
    (df["shutdown_dt"] - df["complaint_dt"]).dt.total_seconds() / 60
)

df["SRT_min_calc"] = (
    (df["restart_dt"] - df["shutdown_dt"]).dt.total_seconds() / 60
)

df["Resolution_Duration_calc"] = (
    (df["restart_dt"] - df["complaint_dt"]).dt.total_seconds() / 60
)

df["Interruption Duration (min)"] = df["Interruption Duration (min)"].replace(0, np.nan)
df["Interruption Duration (min)"] = df["Interruption Duration (min)"].fillna(df["SRT_min_calc"])
df["Interruption Duration (min)"] = df["Interruption Duration (min)"].fillna(0)

df["Resolve Duration"] = df["Resolve Duration"].replace(0, np.nan)
df["Resolve Duration"] = df["Resolve Duration"].fillna(df["Resolution_Duration_calc"])
df["Resolve Duration"] = df["Resolve Duration"].fillna(0)

for col in [
    "Detection_Delay_min",
    "Interruption Duration (min)",
    "Resolve Duration",
    "Affected Consumers",
    "Load Affected (kW)"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    df.loc[df[col] < 0, col] = 0

# ============================================================
# 7. Text Cleaning and Severity Mapping
# ============================================================

text_cols = [
    "Severity Level",
    "Priority Level",
    "Affected Area",
    "Area Type",
    "Fault Category",
    "Fault Type",
    "Reason Category",
    "Reason of Interruption",
    "Status"
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
    else:
        df[col] = "Unknown"

severity_map = {
    "low": 1,
    "minor": 1,
    "medium": 2,
    "moderate": 2,
    "high": 3,
    "major": 3,
    "critical": 4,
    "severe": 4
}

df["Severity_Score"] = (
    df["Severity Level"].astype(str).str.lower().map(severity_map).fillna(1)
)

# ============================================================
# 8. Technical Failure Detection
# ============================================================

technical_keywords = [
    "fault", "trip", "breaker", "transformer", "cable", "feeder",
    "relay", "fuse", "line", "technical", "overload",
    "short circuit", "earth fault", "conductor", "insulator"
]

def is_technical_failure(row):
    text = " ".join([
        str(row.get("Fault Category", "")),
        str(row.get("Fault Type", "")),
        str(row.get("Reason Category", "")),
        str(row.get("Reason of Interruption", ""))
    ]).lower()

    return int(any(k in text for k in technical_keywords))

df["Technical_Failure_Flag"] = df.apply(is_technical_failure, axis=1)

# ============================================================
# 9. Threshold-Based Normalization Functions
# ============================================================

def threshold_cost_normalize(series, threshold):
    s = pd.to_numeric(series, errors="coerce").fillna(0)

    if threshold == 0 or pd.isna(threshold):
        return pd.Series(1.0, index=s.index)

    return (1 - s / threshold).clip(lower=0, upper=1)

def threshold_benefit_normalize(series, threshold):
    s = pd.to_numeric(series, errors="coerce").fillna(0)

    if threshold == 0 or pd.isna(threshold):
        return pd.Series(1.0, index=s.index)

    return (s / threshold).clip(lower=0, upper=1)

# ============================================================
# 10. IEEE / BERC / Utility Thresholds
# ============================================================

THRESHOLDS = {
    "SAIFI_CIFI": 10,
    "SAIDI_CIDI": 600,
    "CAIDI": 120,
    "SRT": 120,
    "DDI": 30,
    "CIM": 1000,
    "LIM": 1000,
    "CRE": 1.0,
    "OSI": 500000,
    "TFER": 0.10,
    "SSII": 1.0
}

# ============================================================
# 11. Classification
# ============================================================

def classify_customer_experience(score):
    if score >= 90:
        return "Excellent Customer Experience"
    elif score >= 75:
        return "Good Customer Experience"
    elif score >= 60:
        return "Moderate Customer Experience"
    elif score >= 40:
        return "Weak Customer Experience"
    else:
        return "Critical Customer Experience"

# ============================================================
# 12. Core Customer Experience Function
# ============================================================

def customer_experience_metrics(g):

    g = g.copy()
    total_events = len(g)

    complaint_count = (
        g["Complaint ID"].nunique(dropna=True)
        if "Complaint ID" in g.columns
        else total_events
    )

    ticket_count = (
        g["Ticket Number"].nunique(dropna=True)
        if "Ticket Number" in g.columns
        else total_events
    )

    t_start = g["complaint_dt"].min()
    t_end = g["restart_dt"].max()

    if pd.notna(t_start) and pd.notna(t_end) and t_end > t_start:
        Tobs_hr = (t_end - t_start).total_seconds() / 3600
    else:
        Tobs_hr = np.nan

    affected_sum = g["Affected Consumers"].sum()
    load_sum = g["Load Affected (kW)"].sum()

    SAIFI_CIFI = affected_sum / complaint_count if complaint_count > 0 else np.nan

    SAIDI_CIDI = (
        (g["Interruption Duration (min)"] * g["Affected Consumers"]).sum()
        / affected_sum
        if affected_sum > 0 else np.nan
    )

    CAIDI = (
        SAIDI_CIDI / SAIFI_CIFI
        if pd.notna(SAIDI_CIDI) and pd.notna(SAIFI_CIFI) and SAIFI_CIFI > 0
        else np.nan
    )

    SRT = g["Interruption Duration (min)"].mean()
    DDI = g["Detection_Delay_min"].mean()

    CIM = affected_sum / total_events if total_events > 0 else np.nan
    LIM = load_sum / total_events if total_events > 0 else np.nan

    status_lower = g["Status"].astype(str).str.lower()

    resolved_rate = status_lower.isin(
        ["closed", "resolved", "completed", "done", "solved"]
    ).mean()

    mean_resolve = g["Resolve Duration"].mean()

    CRE = (
        resolved_rate / (1 + mean_resolve / 60)
        if pd.notna(resolved_rate) and pd.notna(mean_resolve)
        else np.nan
    )

    OSI = (
        (
            g["Severity_Score"]
            * g["Interruption Duration (min)"]
            * g["Affected Consumers"]
            * g["Load Affected (kW)"]
        ).sum() / total_events
        if total_events > 0 else np.nan
    )

    tech_count = g["Technical_Failure_Flag"].sum()

    TFER = (
        tech_count / Tobs_hr
        if pd.notna(Tobs_hr) and Tobs_hr > 0
        else np.nan
    )

    unique_area_count = (
        g["Affected Area"]
        .replace("nan", np.nan)
        .replace("Unknown", np.nan)
        .dropna()
        .nunique()
    )

    SSII = unique_area_count / complaint_count if complaint_count > 0 else np.nan

    return pd.Series({
        "Complaint_Count": complaint_count,
        "Ticket_Count": ticket_count,
        "Observation_Time_hr": Tobs_hr,

        "SAIFI_CIFI": SAIFI_CIFI,
        "SAIDI_CIDI_min": SAIDI_CIDI,
        "CAIDI_min": CAIDI,
        "SRT_min": SRT,
        "DDI_min": DDI,
        "CIM": CIM,
        "LIM_kW": LIM,
        "CRE": CRE,
        "OSI": OSI,
        "TFER": TFER,
        "SSII": SSII,

        "Resolved_Rate": resolved_rate,
        "Mean_Resolve_Duration_min": mean_resolve,
        "Technical_Failure_Count": tech_count,
        "Unique_Affected_Areas": unique_area_count,
        "Total_Affected_Consumers": affected_sum,
        "Total_Load_Affected_kW": load_sum
    })

# ============================================================
# 13. Composite CE Function
# ============================================================

ce_weights = {
    "N_SAIFI_CIFI": 0.15,
    "N_SAIDI_CIDI": 0.15,
    "N_CAIDI": 0.10,
    "N_SRT": 0.10,
    "N_DDI": 0.07,
    "N_CIM": 0.10,
    "N_LIM": 0.08,
    "N_CRE": 0.10,
    "N_OSI": 0.07,
    "N_TFER": 0.04,
    "N_SSII": 0.04
}

def apply_customer_experience_normalization(result):

    result["N_SAIFI_CIFI"] = threshold_cost_normalize(
        result["SAIFI_CIFI"], THRESHOLDS["SAIFI_CIFI"]
    )

    result["N_SAIDI_CIDI"] = threshold_cost_normalize(
        result["SAIDI_CIDI_min"], THRESHOLDS["SAIDI_CIDI"]
    )

    result["N_CAIDI"] = threshold_cost_normalize(
        result["CAIDI_min"], THRESHOLDS["CAIDI"]
    )

    result["N_SRT"] = threshold_cost_normalize(
        result["SRT_min"], THRESHOLDS["SRT"]
    )

    result["N_DDI"] = threshold_cost_normalize(
        result["DDI_min"], THRESHOLDS["DDI"]
    )

    result["N_CIM"] = threshold_cost_normalize(
        result["CIM"], THRESHOLDS["CIM"]
    )

    result["N_LIM"] = threshold_cost_normalize(
        result["LIM_kW"], THRESHOLDS["LIM"]
    )

    result["N_OSI"] = threshold_cost_normalize(
        result["OSI"], THRESHOLDS["OSI"]
    )

    result["N_TFER"] = threshold_cost_normalize(
        result["TFER"], THRESHOLDS["TFER"]
    )

    result["N_SSII"] = threshold_cost_normalize(
        result["SSII"], THRESHOLDS["SSII"]
    )

    result["N_CRE"] = threshold_benefit_normalize(
        result["CRE"], THRESHOLDS["CRE"]
    )

    result["Customer_Experience_Index"] = sum(
        result[col] * weight
        for col, weight in ce_weights.items()
    )

    result["Customer_Experience_Percent"] = (
        result["Customer_Experience_Index"] * 100
    )

    result["Customer_Experience_Class"] = result[
        "Customer_Experience_Percent"
    ].apply(classify_customer_experience)

    return result

# ============================================================
# 14. Year-wise Customer Experience
# ============================================================

year_df = (
    df.groupby("Year", dropna=False)
      .apply(customer_experience_metrics)
      .reset_index()
)

year_df = apply_customer_experience_normalization(year_df)

year_df = year_df.sort_values(
    "Customer_Experience_Percent",
    ascending=False
)

print("\nYEAR-WISE CUSTOMER EXPERIENCE RESULT")
display(year_df.round(2))

# ============================================================
# 15. Feeder-wise Customer Experience
# ============================================================

feeder_df = (
    df.groupby("Feeder Name", dropna=False)
      .apply(customer_experience_metrics)
      .reset_index()
)

feeder_df = apply_customer_experience_normalization(feeder_df)

feeder_df = feeder_df.sort_values(
    "Customer_Experience_Percent",
    ascending=False
)

print("\nFEEDER-WISE CUSTOMER EXPERIENCE RESULT")
display(feeder_df.round(2))

# ============================================================
# 16. Feeder-Year-wise Customer Experience
# ============================================================

feeder_year_df = (
    df.groupby(["Feeder Name", "Year"], dropna=False)
      .apply(customer_experience_metrics)
      .reset_index()
)

feeder_year_df = apply_customer_experience_normalization(feeder_year_df)

feeder_year_df = feeder_year_df.sort_values(
    ["Year", "Customer_Experience_Percent"],
    ascending=[True, False]
)

feeder_year_df["Rank"] = feeder_year_df.groupby("Year")[
    "Customer_Experience_Percent"
].rank(ascending=False, method="dense")

print("\nFEEDER-YEAR-WISE CUSTOMER EXPERIENCE RESULT")
display(feeder_year_df.round(2))

# ============================================================
# 17. Final Display Columns
# ============================================================

final_cols = [
    "Feeder Name",
    "Year",

    "Complaint_Count",
    "Ticket_Count",
    "Observation_Time_hr",

    "SAIFI_CIFI",
    "N_SAIFI_CIFI",

    "SAIDI_CIDI_min",
    "N_SAIDI_CIDI",

    "CAIDI_min",
    "N_CAIDI",

    "SRT_min",
    "N_SRT",

    "DDI_min",
    "N_DDI",

    "CIM",
    "N_CIM",

    "LIM_kW",
    "N_LIM",

    "CRE",
    "N_CRE",

    "OSI",
    "N_OSI",

    "TFER",
    "N_TFER",

    "SSII",
    "N_SSII",

    "Customer_Experience_Index",
    "Customer_Experience_Percent",
    "Customer_Experience_Class",

    "Resolved_Rate",
    "Mean_Resolve_Duration_min",
    "Technical_Failure_Count",
    "Unique_Affected_Areas",
    "Total_Affected_Consumers",
    "Total_Load_Affected_kW",
    "Rank"
]

year_cols = [c for c in final_cols if c in year_df.columns]
feeder_cols = [c for c in final_cols if c in feeder_df.columns]
feeder_year_cols = [c for c in final_cols if c in feeder_year_df.columns]

year_final = year_df[year_cols].copy()
feeder_final = feeder_df[feeder_cols].copy()
feeder_year_final = feeder_year_df[feeder_year_cols].copy()

print("\nYEAR-WISE FINAL RESULT")
display(year_final.round(2))

print("\nFEEDER-WISE FINAL RESULT")
display(feeder_final.round(2))

print("\nFEEDER-YEAR-WISE FINAL RESULT")
display(feeder_year_final.round(2))

# ============================================================
# 18. Save Customer Experience Value for Final Composite Code
# ============================================================

customer_experience_value = feeder_year_df[
    ["Feeder Name", "Year", "Customer_Experience_Percent"]
].copy()

customer_experience_value = customer_experience_value.rename(columns={
    "Customer_Experience_Percent": "Customer_Experience_Percent"
})

print("\nCustomer Experience value saved successfully for final composite code.")
display(customer_experience_value.round(2))

# ============================================================
# 19. Summary
# ============================================================

summary_cols = [
    "Customer_Experience_Index",
    "Customer_Experience_Percent",
    "N_SAIFI_CIFI",
    "N_SAIDI_CIDI",
    "N_CAIDI",
    "N_SRT",
    "N_DDI",
    "N_CIM",
    "N_LIM",
    "N_CRE",
    "N_OSI",
    "N_TFER",
    "N_SSII"
]

summary = feeder_year_df[summary_cols].describe().T.round(4)

print("\nCUSTOMER EXPERIENCE SUMMARY")
display(summary)

# ============================================================
# 20. Export Result
# ============================================================

output_file = "year_wise_feeder_wise_customer_experience.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    year_final.to_excel(writer, sheet_name="Year_CE", index=False)
    feeder_final.to_excel(writer, sheet_name="Feeder_CE", index=False)
    feeder_year_final.to_excel(writer, sheet_name="Feeder_Year_CE", index=False)
    customer_experience_value.to_excel(writer, sheet_name="CE_Value", index=False)
    summary.to_excel(writer, sheet_name="Summary")

print("\nCustomer Experience calculation completed successfully.")
print("Saved:", output_file)

# Optional download
# files.download(output_file)

Saving Updated 5000 Complaints Sreepur.xlsx to Updated 5000 Complaints Sreepur (17).xlsx
File loaded successfully
Total records: 5000


,Complaint ID,Ticket Number,Complainant Name,Phone Number,Complaintn Receiver,Complaint Date,Complaint Time,Shutdown Date,Shutdown Time,Restart Date,...,Affected Consumers,Load Affected (kW),Affected Area.1,Reason of Interruption,Reason Category,Reason of Interruption.1,Fault Category,Interruption Description,Action Taken,Fault Type
0,C05952,TKT-05952,সোহেল খান,1563963655,রফিক (EMP412),01-Jan-2022,03:02,01-Jan-2022,03:06,01-Jan-2022,...,305,1716,কাজী তলা বাজার,ট্রান্সফরমার বিকল হওয়ায় সম্পূর্ণ এলাকায় বিদ...,Transformer Fault,Transformer Failure,Transformer Fault,ট্রান্সফরমার বিকল হওয়ায় সম্পূর্ণ এলাকায় বিদ...,ত্রুটি সংশোধন করে পুনঃসংযোগ দেওয়া হয়েছে,Transformer Fault
1,C00522,TKT-00522,মমিন বেপারী,1273201828,হাসান (EMP225),01-Jan-2022,03:30,01-Jan-2022,03:37,01-Jan-2022,...,216,3183,মুলাইদ ব্লক,ট্রান্সফরমার বিকল হওয়ায় সম্পূর্ণ এলাকায় বিদ...,Transformer Fault,Transformer Failure,Transformer Fault,ট্রান্সফরমার বিকল হওয়ায় সম্পূর্ণ এলাকায় বিদ...,লাইন মেরামত করে চালু করা হয়েছে,Transformer Fault
2,C00669,TKT-00669,সালাম মল্লিক,1194051674,শরিফ (EMP318),01-Jan-2022,10:30,01-Jan-2022,10:40,01-Jan-2022,...,342,3487,"রংগেলা বাজার সংলগ্ন এলাকা, তার ছিঁড়ে গেছে",বজ্রপাতের কারণে ফিডার ট্রিপ করে সরবরাহ বন্ধ হয়,Weather Fault,Lightning Fault,Weather Fault,বজ্রপাতের কারণে ফিডার ট্রিপ করে সরবরাহ বন্ধ হয়,লাইন মেরামত করে চালু করা হয়েছে,Weather Fault
3,C04133,TKT-04133,আব্দুল শেখ,1024619306,নাঈম (EMP390),01-Jan-2022,14:53,01-Jan-2022,14:51,01-Jan-2022,...,609,3382,বৈরাইদের চালা পাড়া,লোড অসমতার কারণে ভোল্টেজ স্বাভাবিকের নিচে নেমে...,Voltage Issue,Voltage Drop,Voltage Issue,লোড অসমতার কারণে ভোল্টেজ স্বাভাবিকের নিচে নেমে...,লোড ব্যালেন্স করা হয়েছে,Voltage Issue
4,C05386,TKT-05386,হাসান খান,1994956873,নাঈম (EMP390),01-Jan-2022,16:04,01-Jan-2022,16:11,01-Jan-2022,...,380,1810,"মাওনা চৌরাস্তা সংলগ্ন এলাকা, মাওনা বাসস্ট্যান্...",বজ্রপাতের কারণে ফিডার ট্রিপ করে সরবরাহ বন্ধ হয়,Weather Fault,Lightning Fault,Weather Fault,বজ্রপাতের কারণে ফিডার ট্রিপ করে সরবরাহ বন্ধ হয়,যন্ত্রাংশ পরিবর্তন করা হয়েছে,Weather Fault



YEAR-WISE CUSTOMER EXPERIENCE RESULT


/tmp/ipykernel_21811/1801507237.py:444: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(customer_experience_metrics)


,Year,Complaint_Count,Ticket_Count,Observation_Time_hr,SAIFI_CIFI,SAIDI_CIDI_min,CAIDI_min,SRT_min,DDI_min,CIM,...,N_DDI,N_CIM,N_LIM,N_OSI,N_TFER,N_SSII,N_CRE,Customer_Experience_Index,Customer_Experience_Percent,Customer_Experience_Class
1,2023,1255.0,1255.0,8754.25,498.69,121.03,0.24,122.91,3.31,498.69,...,0.89,0.5,0.0,0.0,0.22,0.40,0.32,0.39,38.91,Critical Customer Experience
2,2024,1279.0,1279.0,8781.73,496.60,118.19,0.24,120.33,3.46,496.60,...,0.88,0.5,0.0,0.0,0.20,0.38,0.33,0.39,38.86,Critical Customer Experience
0,2022,1223.0,1223.0,8748.67,503.05,122.94,0.24,123.33,3.54,503.05,...,0.88,0.5,0.0,0.0,0.22,0.38,0.32,0.39,38.70,Critical Customer Experience
3,2025,1243.0,1243.0,8752.75,499.71,125.75,0.25,124.83,3.44,499.71,...,0.89,0.5,0.0,0.0,0.20,0.38,0.32,0.39,38.54,Critical Customer Experience


...


Customer Experience value saved successfully for final composite code.


,Feeder Name,Year,Customer_Experience_Percent
4,Feeder-10,2022,42.27
40,Feeder-9,2022,41.94
24,Feeder-5,2022,41.80
32,Feeder-7,2022,41.69
12,Feeder-2,2022,41.62
20,Feeder-4,2022,41.48
16,Feeder-3,2022,41.19
0,Feeder-1,2022,41.10
36,Feeder-8,2022,41.07
28,Feeder-6,2022,40.75



CUSTOMER EXPERIENCE SUMMARY


,count,mean,std,min,25%,50%,75%,max
Customer_Experience_Index,44.0,0.4164,0.0079,0.4055,0.4109,0.4146,0.4198,0.4354
Customer_Experience_Percent,44.0,41.6368,0.7920,40.5527,41.0921,41.4582,41.9757,43.5392
N_SAIFI_CIFI,44.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
N_SAIDI_CIDI,44.0,0.7968,0.0182,0.7540,0.7823,0.7973,0.8105,0.8375
N_CAIDI,44.0,0.9980,0.0002,0.9976,0.9978,0.9979,0.9981,0.9985
N_SRT,44.0,0.0246,0.0400,0.0000,0.0000,0.0000,0.0437,0.1689
N_DDI,44.0,0.8853,0.0090,0.8685,0.8788,0.8866,0.8917,0.9045
N_CIM,44.0,0.5008,0.0173,0.4592,0.4899,0.5026,0.5128,0.5326
N_LIM,44.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
N_CRE,44.0,0.3247,0.0186,0.2890,0.3120,0.3266,0.3382,0.3693



Customer Experience calculation completed successfully.
Saved: year_wise_feeder_wise_customer_experience.xlsx


# **Stability**

In [ ]:
# ============================================================
# FINAL COMPOSITE STABILITY INDEX
# Sfinal = 0.25SV + 0.20SI + 0.20Sf + 0.15SPF + 0.10SL + 0.10SE
# Every sub-parameter is normalized before composite calculation
# Google Colab Compatible
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files

# ============================================================
# 1. FILE UPLOAD
# ============================================================

print("Upload your Excel/CSV file:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

if file_name.lower().endswith(".csv"):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name)

df.columns = [str(c).strip() for c in df.columns]

print("Loaded:", file_name)
print("Shape:", df.shape)
display(df.head())


# ============================================================
# 2. ENGINEERING LIMITS
# ============================================================

V_NOMINAL = 63.5
V_ALLOWED_DEV = 0.10 * V_NOMINAL

F_NOMINAL = 50.0
F_ALLOWED_DEV = 0.5

PF_TARGET = 0.95

THDV_MAX = 5.0
THDI_MAX = 8.0

VUF_MAX = 0.05
CUF_MAX = 0.10

ROCOF_MAX = 0.20
TRANSIENT_MAX = 10.0
FLICKER_MAX = 1.0

SI_MAX = 1.0
FES_MAX = 1.0
OF_MAX = 1.0
UF_MAX = 1.0

PF_UNBALANCE_MAX = 0.10
PF_STD_MAX = 0.05

Q_RATIO_MAX = 0.50
Q_FACTOR_MAX = 0.50

EP_UNBALANCE_MAX = 0.10
ES_UNBALANCE_MAX = 0.10


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def to_num(x):
    return pd.to_numeric(x, errors="coerce")


def clamp(x, low=0.0, high=1.0):
    if isinstance(x, pd.Series):
        return x.clip(lower=low, upper=high)
    return pd.Series(x).clip(lower=low, upper=high)


def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def safe_divide(a, b, default=0.0):
    a = to_num(a)
    b = to_num(b)
    out = a / b.replace(0, np.nan)
    return out.replace([np.inf, -np.inf], np.nan).fillna(default)


def cost_norm(x, xmax):
    """
    Cost-type normalization:
    NX = 1 - X / Xmax
    """
    x = to_num(x).abs()

    if xmax is None or pd.isna(xmax) or xmax == 0:
        xmax = x.max()

    if pd.isna(xmax) or xmax == 0:
        return pd.Series(1.0, index=df.index)

    return clamp(1 - x / xmax)


def benefit_norm(x, xmax):
    """
    Benefit-type normalization:
    NX = X / Xmax
    """
    x = to_num(x).abs()

    if xmax is None or pd.isna(xmax) or xmax == 0:
        xmax = x.max()

    if pd.isna(xmax) or xmax == 0:
        return pd.Series(0.0, index=df.index)

    return clamp(x / xmax)


def nominal_norm(x, nominal, allowed_dev):
    """
    Nominal-type normalization:
    NX = 1 - |X - Xnom| / ΔXallowed
    """
    x = to_num(x)
    return clamp(1 - abs(x - nominal) / allowed_dev)


def weighted_sum(data, weights):
    available_cols = [
        col for col in weights
        if col in data.columns and data[col].notna().sum() > 0
    ]

    total_weight = sum(weights[col] for col in available_cols)

    if total_weight == 0:
        return pd.Series(np.nan, index=data.index)

    score = pd.Series(0.0, index=data.index)

    for col in available_cols:
        score += data[col].fillna(0) * weights[col] / total_weight

    return clamp(score)


# ============================================================
# 4. TIMESTAMP HANDLING
# ============================================================

time_col = find_col(df, ["Time Stamp", "Timestamp", "DateTime", "Datetime", "time", "Time"])

if time_col is None and all(c in df.columns for c in ["Date", "Time"]):
    df["Timestamp"] = pd.to_datetime(
        df["Date"].astype(str) + " " + df["Time"].astype(str),
        errors="coerce"
    )
    time_col = "Timestamp"

if time_col is not None:
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df = df.sort_values(time_col).reset_index(drop=True)


# ============================================================
# 5. VOLTAGE STABILITY SUB-PARAMETERS
# Notch, Flicker, Transient = Event-Based
# ============================================================

voltage_cols = [c for c in ["UA", "UB", "UC"] if c in df.columns]

if len(voltage_cols) == 3:
    voltage_data = df[voltage_cols].apply(to_num)
    df["V_mean"] = voltage_data.mean(axis=1)
    df["V_max"] = voltage_data.max(axis=1)
    df["V_min"] = voltage_data.min(axis=1)
    df["Voltage_Variability"] = voltage_data.std(axis=1, ddof=0)
elif "UAvg" in df.columns:
    df["V_mean"] = to_num(df["UAvg"])
    df["V_max"] = df["V_mean"]
    df["V_min"] = df["V_mean"]
    df["Voltage_Variability"] = 0.0
else:
    df["V_mean"] = np.nan
    df["V_max"] = np.nan
    df["V_min"] = np.nan
    df["Voltage_Variability"] = np.nan

df["Voltage_Deviation"] = abs(df["V_mean"] - V_NOMINAL) / V_NOMINAL

df["SagSeverity"] = np.where(
    df["V_min"] < V_NOMINAL - V_ALLOWED_DEV,
    (V_NOMINAL - df["V_min"]) / V_NOMINAL,
    0.0
)

df["SwellSeverity"] = np.where(
    df["V_max"] > V_NOMINAL + V_ALLOWED_DEV,
    (df["V_max"] - V_NOMINAL) / V_NOMINAL,
    0.0
)

df["Sustained_Interruption"] = np.where(
    df["V_mean"] < 0.10 * V_NOMINAL,
    1.0,
    0.0
)

if len(voltage_cols) == 3:
    df["VUF"] = (
        voltage_data
        .sub(df["V_mean"], axis=0)
        .abs()
        .max(axis=1) / df["V_mean"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)
else:
    df["VUF"] = 0.0

if "UTHAvg" in df.columns:
    df["THDv"] = to_num(df["UTHAvg"])
elif all(c in df.columns for c in ["UTHA", "UTHB", "UTHC"]):
    df["THDv"] = df[["UTHA", "UTHB", "UTHC"]].apply(to_num).mean(axis=1)
else:
    df["THDv"] = 0.0


# ============================================================
# EVENT-BASED NOTCH, FLICKER, TRANSIENT
# ============================================================

NOTCH_THD_THRESHOLD = 8.0
PST_LIMIT = 1.0
PLT_LIMIT = 0.8
TRANSIENT_LIMIT = 10.0

pst_col = find_col(df, ["Pst", "PST", "pst"])
plt_col = find_col(df, ["Plt", "PLT", "plt"])

df["Pst_used"] = to_num(df[pst_col]) if pst_col else 0.0
df["Plt_used"] = to_num(df[plt_col]) if plt_col else 0.0

# Flicker event: 1 = event occurred, 0 = no event
df["Pst_Event"] = (df["Pst_used"] > PST_LIMIT).astype(int) if pst_col else 0
df["Plt_Event"] = (df["Plt_used"] > PLT_LIMIT).astype(int) if plt_col else 0
df["Flicker_Event"] = ((df["Pst_Event"] == 1) | (df["Plt_Event"] == 1)).astype(int)

# Transient event based on rate of voltage change
if time_col is not None:
    dt = df[time_col].diff().dt.total_seconds()
    dV = df["V_mean"].diff()
    df["ROVC"] = (
        dV / dt.replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)
else:
    df["ROVC"] = 0.0

df["Transient_Event"] = (df["ROVC"].abs() > TRANSIENT_LIMIT).astype(int)

# Notch event using THDv threshold
df["Notch_Event"] = (df["THDv"] > NOTCH_THD_THRESHOLD).astype(int)


# ============================================================
# VSI and VCR
# ============================================================

df["VSI"] = clamp(1 - ((df["V_max"] - df["V_min"]) / V_NOMINAL))

df["Voltage_Compliance_Flag"] = (
    (df["V_mean"] >= V_NOMINAL - V_ALLOWED_DEV) &
    (df["V_mean"] <= V_NOMINAL + V_ALLOWED_DEV)
).astype(int)

df["VCR"] = df["Voltage_Compliance_Flag"].expanding().mean() * 100


# ============================================================
# VOLTAGE NORMALIZATION
# Event-based terms use: N = 1 - Event
# ============================================================

df["NV"] = nominal_norm(df["V_mean"], V_NOMINAL, V_ALLOWED_DEV)
df["NVD"] = cost_norm(df["Voltage_Deviation"], df["Voltage_Deviation"].max())
df["NSag"] = cost_norm(df["SagSeverity"], df["SagSeverity"].max())
df["NSwell"] = cost_norm(df["SwellSeverity"], df["SwellSeverity"].max())
df["NSI"] = cost_norm(df["Sustained_Interruption"], SI_MAX)
df["NVUF"] = cost_norm(df["VUF"], VUF_MAX)
df["NVvar"] = cost_norm(df["Voltage_Variability"], df["Voltage_Variability"].max())
df["NTHDv"] = cost_norm(df["THDv"], THDV_MAX)

# Event-based normalization
df["NFlicker_Event"] = 1.0 - df["Flicker_Event"]
df["NTransient_Event"] = 1.0 - df["Transient_Event"]
df["NNotch_Event"] = 1.0 - df["Notch_Event"]

df["NVSI"] = benefit_norm(df["VSI"], 1.0)
df["NVCR"] = clamp(df["VCR"] / 100)


# ============================================================
# VOLTAGE COMPOSITE
# ============================================================

voltage_weights = {
    "NV": 0.12,
    "NVD": 0.08,
    "NSag": 0.08,
    "NSwell": 0.08,
    "NSI": 0.08,
    "NVUF": 0.08,
    "NVvar": 0.06,
    "NTHDv": 0.12,
    "NFlicker_Event": 0.08,
    "NTransient_Event": 0.06,
    "NNotch_Event": 0.06,
    "NVSI": 0.07,
    "NVCR": 0.05
}

df["Voltage_Composite"] = weighted_sum(df, voltage_weights)


# ============================================================
# 6. CURRENT STABILITY SUB-PARAMETERS
# ============================================================

current_cols = [c for c in ["IA", "IB", "IC"] if c in df.columns]

if len(current_cols) == 3:
    current_data = df[current_cols].apply(to_num)
    df["I_mean"] = current_data.mean(axis=1)
    df["I_max"] = current_data.max(axis=1)
    df["I_min"] = current_data.min(axis=1)
    df["Current_Variability"] = current_data.std(axis=1, ddof=0)

    df["CUF"] = (
        current_data
        .sub(df["I_mean"], axis=0)
        .abs()
        .max(axis=1) / df["I_mean"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)

else:
    df["I_mean"] = to_num(df["IAvg"]) if "IAvg" in df.columns else np.nan
    df["I_max"] = df["I_mean"]
    df["I_min"] = df["I_mean"]
    df["Current_Variability"] = 0.0
    df["CUF"] = 0.0

df["Neutral_Current"] = to_num(df["IN"]).abs() if "IN" in df.columns else 0.0

if "ITHAvg" in df.columns:
    df["THDi"] = to_num(df["ITHAvg"])
elif all(c in df.columns for c in ["ITHA", "ITHB", "ITHC"]):
    df["THDi"] = df[["ITHA", "ITHB", "ITHC"]].apply(to_num).mean(axis=1)
else:
    df["THDi"] = 0.0

harmonic_cols = [
    c for c in df.columns
    if c.startswith(("ITHX", "ITHY", "ITHZ", "ITHV", "ITHW"))
]

df["Harmonic_Current"] = (
    df[harmonic_cols].apply(to_num).mean(axis=1)
    if harmonic_cols else df["THDi"]
)

df["Current_Imbalance_Ratio"] = df["CUF"]


# -----------------------------
# Current Normalization
# -----------------------------

df["NI"] = cost_norm(df["I_mean"], df["I_mean"].max())
df["NCUF"] = cost_norm(df["CUF"], CUF_MAX)
df["NIN"] = cost_norm(df["Neutral_Current"], df["Neutral_Current"].max())
df["NTHDi"] = cost_norm(df["THDi"], THDI_MAX)
df["NIh"] = cost_norm(df["Harmonic_Current"], df["Harmonic_Current"].max())
df["NIvar"] = cost_norm(df["Current_Variability"], df["Current_Variability"].max())
df["NCIR"] = cost_norm(df["Current_Imbalance_Ratio"], df["Current_Imbalance_Ratio"].max())


current_weights = {
    "NI": 0.18,
    "NCUF": 0.16,
    "NIN": 0.14,
    "NTHDi": 0.18,
    "NIh": 0.14,
    "NIvar": 0.10,
    "NCIR": 0.10
}

df["Current_Composite"] = weighted_sum(df, current_weights)


# ============================================================
# 7. FREQUENCY STABILITY SUB-PARAMETERS
# ============================================================

freq_cols = [c for c in ["FA", "FB", "FC"] if c in df.columns]

if len(freq_cols) == 3:
    freq_data = df[freq_cols].apply(to_num)
    df["F_mean"] = freq_data.mean(axis=1)
    df["F_max"] = freq_data.max(axis=1)
    df["F_min"] = freq_data.min(axis=1)
    df["Frequency_Variability"] = freq_data.std(axis=1, ddof=0)

    df["Frequency_Unbalance"] = (
        freq_data
        .sub(df["F_mean"], axis=0)
        .abs()
        .max(axis=1) / df["F_mean"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)

elif "FAvg" in df.columns:
    df["F_mean"] = to_num(df["FAvg"])
    df["F_max"] = df["F_mean"]
    df["F_min"] = df["F_mean"]
    df["Frequency_Variability"] = 0.0
    df["Frequency_Unbalance"] = 0.0
else:
    df["F_mean"] = np.nan
    df["F_max"] = np.nan
    df["F_min"] = np.nan
    df["Frequency_Variability"] = np.nan
    df["Frequency_Unbalance"] = np.nan

df["Frequency_Deviation"] = abs(df["F_mean"] - F_NOMINAL)
df["Frequency_Swing"] = df["F_max"] - df["F_min"]

if time_col is not None:
    dt = df[time_col].diff().dt.total_seconds()
    dF = df["F_mean"].diff()
    df["RoCoF"] = (
        dF / dt.replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0).abs()
else:
    df["RoCoF"] = 0.0

df["Over_Frequency_Event"] = np.where(
    df["F_mean"] > F_NOMINAL + F_ALLOWED_DEV,
    1.0,
    0.0
)

df["Under_Frequency_Event"] = np.where(
    df["F_mean"] < F_NOMINAL - F_ALLOWED_DEV,
    1.0,
    0.0
)

df["FES"] = (
    df["Frequency_Deviation"] / F_ALLOWED_DEV +
    df["RoCoF"] / ROCOF_MAX
) / 2

df["FSI"] = 1 - (
    0.40 * df["Frequency_Deviation"] / F_ALLOWED_DEV +
    0.30 * df["RoCoF"] / ROCOF_MAX +
    0.30 * df["Frequency_Swing"] / max(df["Frequency_Swing"].max(), 1)
)

df["FSI"] = clamp(df["FSI"])


# -----------------------------
# Frequency Normalization
# -----------------------------

df["Nf"] = nominal_norm(df["F_mean"], F_NOMINAL, F_ALLOWED_DEV)
df["NDelta_f"] = cost_norm(df["Frequency_Deviation"], F_ALLOWED_DEV)
df["Nfvar"] = cost_norm(df["Frequency_Variability"], df["Frequency_Variability"].max())
df["Nfunb"] = cost_norm(df["Frequency_Unbalance"], df["Frequency_Unbalance"].max())
df["NRoCoF"] = cost_norm(df["RoCoF"], ROCOF_MAX)
df["NFswing"] = cost_norm(df["Frequency_Swing"], df["Frequency_Swing"].max())
df["NFES"] = cost_norm(df["FES"], FES_MAX)
df["NOF"] = cost_norm(df["Over_Frequency_Event"], 1.0)
df["NUF"] = cost_norm(df["Under_Frequency_Event"], 1.0)
df["NFSI"] = benefit_norm(df["FSI"], 1.0)


frequency_weights = {
    "Nf": 0.14,
    "NDelta_f": 0.12,
    "Nfvar": 0.10,
    "Nfunb": 0.08,
    "NRoCoF": 0.14,
    "NFswing": 0.10,
    "NFES": 0.10,
    "NOF": 0.07,
    "NUF": 0.07,
    "NFSI": 0.08
}

df["Frequency_Composite"] = weighted_sum(df, frequency_weights)


# ============================================================
# 8. POWER FACTOR STABILITY SUB-PARAMETERS
# ============================================================

pf_cols = [c for c in ["PFA", "PFB", "PFC"] if c in df.columns]

if len(pf_cols) == 3:
    pf_data = df[pf_cols].apply(to_num).abs()
    df["PF_mean"] = pf_data.mean(axis=1)
    df["PF_std"] = pf_data.std(axis=1, ddof=0)

    df["PF_unbalance"] = (
        pf_data
        .sub(df["PF_mean"], axis=0)
        .abs()
        .max(axis=1) / df["PF_mean"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)

elif "PFAvg" in df.columns:
    df["PF_mean"] = to_num(df["PFAvg"]).abs()
    df["PF_std"] = 0.0
    df["PF_unbalance"] = 0.0
elif all(c in df.columns for c in ["PSum", "SSum"]):
    df["PF_mean"] = safe_divide(df["PSum"].abs(), df["SSum"].abs())
    df["PF_std"] = 0.0
    df["PF_unbalance"] = 0.0
else:
    df["PF_mean"] = np.nan
    df["PF_std"] = np.nan
    df["PF_unbalance"] = np.nan

df["PF_deviation"] = abs(PF_TARGET - df["PF_mean"])

if all(c in df.columns for c in ["PSum", "SSum"]):
    df["PF_calc"] = safe_divide(df["PSum"].abs(), df["SSum"].abs())
    df["P_ratio"] = df["PF_calc"]
else:
    df["PF_calc"] = df["PF_mean"]
    df["P_ratio"] = df["PF_mean"]

if all(c in df.columns for c in ["QSum", "SSum"]):
    df["Q_ratio"] = safe_divide(df["QSum"].abs(), df["SSum"].abs())
else:
    df["Q_ratio"] = 0.0

if all(c in df.columns for c in ["QSum", "PSum"]):
    df["Q_factor"] = safe_divide(df["QSum"].abs(), df["PSum"].abs())
else:
    df["Q_factor"] = 0.0


# -----------------------------
# Power Factor Normalization
# -----------------------------

df["NPF"] = benefit_norm(df["PF_mean"], PF_TARGET)
df["NPF_dev"] = cost_norm(df["PF_deviation"], 1 - PF_TARGET)
df["NPF_unbalance"] = cost_norm(df["PF_unbalance"], PF_UNBALANCE_MAX)
df["NPF_std"] = cost_norm(df["PF_std"], PF_STD_MAX)
df["NPF_calc"] = benefit_norm(df["PF_calc"], PF_TARGET)
df["NP_ratio"] = benefit_norm(df["P_ratio"], 1.0)
df["NQ_ratio"] = cost_norm(df["Q_ratio"], Q_RATIO_MAX)
df["NQ_factor"] = cost_norm(df["Q_factor"], Q_FACTOR_MAX)


pf_weights = {
    "NPF": 0.20,
    "NPF_dev": 0.15,
    "NPF_unbalance": 0.15,
    "NPF_std": 0.10,
    "NPF_calc": 0.15,
    "NP_ratio": 0.10,
    "NQ_ratio": 0.10,
    "NQ_factor": 0.05
}

df["Power_Factor_Composite"] = weighted_sum(df, pf_weights)


# ============================================================
# 9. LOAD STABILITY SUB-PARAMETERS
# ============================================================

if all(c in df.columns for c in ["PA", "PB", "PC"]):
    p_data = df[["PA", "PB", "PC"]].apply(to_num).abs()
    df["Active_Power"] = p_data.sum(axis=1)
    df["Load_Variability"] = p_data.std(axis=1, ddof=0)

    df["Load_Unbalance"] = (
        p_data
        .sub(p_data.mean(axis=1), axis=0)
        .abs()
        .max(axis=1) / p_data.mean(axis=1).replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)

elif "PSum" in df.columns:
    df["Active_Power"] = to_num(df["PSum"]).abs()
    df["Load_Variability"] = df["Active_Power"].rolling(3, min_periods=1).std().fillna(0.0)
    df["Load_Unbalance"] = 0.0
else:
    df["Active_Power"] = np.nan
    df["Load_Variability"] = np.nan
    df["Load_Unbalance"] = np.nan

df["Reactive_Power"] = to_num(df["QSum"]).abs() if "QSum" in df.columns else 0.0
df["Apparent_Power"] = to_num(df["SSum"]).abs() if "SSum" in df.columns else df["Active_Power"]

df["NP"] = cost_norm(df["Active_Power"], df["Active_Power"].max())
df["NQ"] = cost_norm(df["Reactive_Power"], df["Reactive_Power"].max())
df["NS"] = cost_norm(df["Apparent_Power"], df["Apparent_Power"].max())
df["NLV"] = cost_norm(df["Load_Variability"], df["Load_Variability"].max())
df["NLU"] = cost_norm(df["Load_Unbalance"], df["Load_Unbalance"].max())


load_weights = {
    "NP": 0.28,
    "NQ": 0.22,
    "NS": 0.25,
    "NLV": 0.15,
    "NLU": 0.10
}

df["Load_Composite"] = weighted_sum(df, load_weights)


# ============================================================
# 10. ENERGY STABILITY SUB-PARAMETERS
# ============================================================

df["EP_total"] = to_num(df["EPSum"]).abs() if "EPSum" in df.columns else np.nan
df["EQ_total"] = to_num(df["EQSum"]).abs() if "EQSum" in df.columns else 0.0
df["ES_total"] = to_num(df["ESSum"]).abs() if "ESSum" in df.columns else df["EP_total"]

df["EP_to_ES"] = safe_divide(df["EP_total"], df["ES_total"])
df["EQ_to_ES"] = safe_divide(df["EQ_total"], df["ES_total"])
df["EQ_to_EP"] = safe_divide(df["EQ_total"], df["EP_total"])

if all(c in df.columns for c in ["EPA", "EPB", "EPC"]):
    ep_data = df[["EPA", "EPB", "EPC"]].apply(to_num).abs()
    df["EP_phase_mean"] = ep_data.mean(axis=1)

    df["EP_unbalance"] = (
        ep_data
        .sub(df["EP_phase_mean"], axis=0)
        .abs()
        .max(axis=1) / df["EP_phase_mean"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)
else:
    df["EP_unbalance"] = 0.0

if all(c in df.columns for c in ["ESA", "ESB", "ESC"]):
    es_data = df[["ESA", "ESB", "ESC"]].apply(to_num).abs()
    df["ES_phase_mean"] = es_data.mean(axis=1)

    df["ES_unbalance"] = (
        es_data
        .sub(df["ES_phase_mean"], axis=0)
        .abs()
        .max(axis=1) / df["ES_phase_mean"].replace(0, np.nan)
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)
else:
    df["ES_unbalance"] = 0.0

df["Energy_Not_Supplied"] = abs(df["ES_total"] - df["EP_total"])


# -----------------------------
# Energy Normalization
# -----------------------------

df["NEP_to_ES"] = benefit_norm(df["EP_to_ES"], 1.0)
df["NEQ_to_ES"] = cost_norm(df["EQ_to_ES"], Q_RATIO_MAX)
df["NEQ_to_EP"] = cost_norm(df["EQ_to_EP"], Q_RATIO_MAX)
df["NEP_unbalance"] = cost_norm(df["EP_unbalance"], EP_UNBALANCE_MAX)
df["NES_unbalance"] = cost_norm(df["ES_unbalance"], ES_UNBALANCE_MAX)
df["NENS"] = cost_norm(df["Energy_Not_Supplied"], df["Energy_Not_Supplied"].max())


energy_weights = {
    "NEP_to_ES": 0.25,
    "NEQ_to_ES": 0.18,
    "NEQ_to_EP": 0.18,
    "NEP_unbalance": 0.14,
    "NES_unbalance": 0.10,
    "NENS": 0.15
}

df["Energy_Composite"] = weighted_sum(df, energy_weights)


# ============================================================
# 11. FINAL COMPOSITE STABILITY INDEX
# Sfinal = 0.25SV + 0.20SI + 0.20Sf + 0.15SPF + 0.10SL + 0.10SE
# ============================================================

final_weights = {
    "Voltage_Composite": 0.25,
    "Current_Composite": 0.20,
    "Frequency_Composite": 0.20,
    "Power_Factor_Composite": 0.15,
    "Load_Composite": 0.10,
    "Energy_Composite": 0.10
}

df["Final_Composite_Stability_Index"] = weighted_sum(df, final_weights)
df["Final_Composite_Stability_Percent"] = df["Final_Composite_Stability_Index"] * 100


def classify_stability(x):
    if x >= 90:
        return "Excellent Stability"
    elif x >= 75:
        return "Stable"
    elif x >= 60:
        return "Moderate Stability"
    elif x >= 40:
        return "Weak Stability"
    else:
        return "Critical Stability"


df["Final_Stability_Class"] = df["Final_Composite_Stability_Percent"].apply(classify_stability)


# ============================================================
# 12. SHOW ALL SUB-PARAMETERS AND NORMALIZED PARAMETERS
# ============================================================

voltage_feature_cols = [
    "V_mean", "V_max", "V_min",
    "Voltage_Deviation", "SagSeverity", "SwellSeverity",
    "Sustained_Interruption", "VUF", "Voltage_Variability",
    "THDv", "Pst_used", "Plt_used", "Pst_Event", "Plt_Event",
    "Flicker_Event", "ROVC", "Transient_Event", "Notch_Event",
    "VSI", "VCR",
    "Voltage_Composite"
]

current_feature_cols = [
    "I_mean", "I_max", "I_min", "Current_Variability", "CUF",
    "Neutral_Current", "THDi", "Harmonic_Current",
    "Current_Imbalance_Ratio",
    "Current_Composite"
]

frequency_feature_cols = [
    "F_mean", "Frequency_Deviation", "Frequency_Variability",
    "Frequency_Unbalance", "RoCoF", "Frequency_Swing",
    "FES", "Over_Frequency_Event", "Under_Frequency_Event",
    "FSI",
    "Frequency_Composite"
]

pf_feature_cols = [
    "PF_mean", "PF_deviation", "PF_unbalance", "PF_std",
    "PF_calc", "P_ratio", "Q_ratio", "Q_factor",
    "Power_Factor_Composite"
]

load_feature_cols = [
    "Active_Power", "Reactive_Power", "Apparent_Power",
    "Load_Variability", "Load_Unbalance",
    "Load_Composite"
]

energy_feature_cols = [
    "EP_total", "EQ_total", "ES_total",
    "EP_to_ES", "EQ_to_ES", "EQ_to_EP",
    "EP_unbalance", "ES_unbalance", "Energy_Not_Supplied",
    "Energy_Composite"
]

final_feature_cols = [
    "Voltage_Composite", "Current_Composite", "Frequency_Composite",
    "Power_Factor_Composite", "Load_Composite", "Energy_Composite",
    "Final_Composite_Stability_Index",
    "Final_Composite_Stability_Percent",
    "Final_Stability_Class"
]


def existing_cols(cols):
    return [c for c in cols if c in df.columns]

'''
print("\nVOLTAGE SUB-PARAMETERS + NORMALIZED SCORES")
display(df[existing_cols(voltage_feature_cols)].head(20))

print("\nCURRENT SUB-PARAMETERS + NORMALIZED SCORES")
display(df[existing_cols(current_feature_cols)].head(20))

print("\nFREQUENCY SUB-PARAMETERS + NORMALIZED SCORES")
display(df[existing_cols(frequency_feature_cols)].head(20))

print("\nPOWER FACTOR SUB-PARAMETERS + NORMALIZED SCORES")
display(df[existing_cols(pf_feature_cols)].head(20))

print("\nLOAD SUB-PARAMETERS + NORMALIZED SCORES")
display(df[existing_cols(load_feature_cols)].head(20))

print("\nENERGY SUB-PARAMETERS + NORMALIZED SCORES")
display(df[existing_cols(energy_feature_cols)].head(20))

print("\nFINAL COMPOSITE STABILITY CONTRIBUTION")
display(df[existing_cols(final_feature_cols)].head(20))
'''

# ============================================================
# 13. SUMMARY TABLES
# ============================================================

voltage_summary = df[existing_cols(voltage_feature_cols)].describe().T.round(4)
current_summary = df[existing_cols(current_feature_cols)].describe().T.round(4)
frequency_summary = df[existing_cols(frequency_feature_cols)].describe().T.round(4)
pf_summary = df[existing_cols(pf_feature_cols)].describe().T.round(4)
load_summary = df[existing_cols(load_feature_cols)].describe().T.round(4)
energy_summary = df[existing_cols(energy_feature_cols)].describe().T.round(4)
final_summary = df[existing_cols(final_feature_cols[:-1])].describe().T.round(4)

print("\nFINAL COMPOSITE STABILITY SUMMARY")
display(final_summary)


# ============================================================
# 14. SAVE OUTPUT
# ============================================================

output_file = "Full_Normalized_Final_Composite_Stability.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="All_Features", index=False)

    voltage_summary.to_excel(writer, sheet_name="Voltage_Summary")
    current_summary.to_excel(writer, sheet_name="Current_Summary")
    frequency_summary.to_excel(writer, sheet_name="Frequency_Summary")
    pf_summary.to_excel(writer, sheet_name="PF_Summary")
    load_summary.to_excel(writer, sheet_name="Load_Summary")
    energy_summary.to_excel(writer, sheet_name="Energy_Summary")
    final_summary.to_excel(writer, sheet_name="Final_Stability_Summary")

print("\nSaved:", output_file)
#files.download(output_file)



# ============================================================
# SAVE STABILITY VALUE FOR FINAL COMPOSITE CODE
# ============================================================
# First stability dataset is considered as Feeder 1 stability

stability_value = pd.DataFrame({
    "Feeder Name": ["Feeder 1"],
    "Stability_Percent": [
        df["Final_Composite_Stability_Percent"].mean()
    ]
})

print("Stability value saved successfully.")
display(stability_value.round(2))

Upload your Excel/CSV file:


Saving Power Logger Data Sreepur 123.xlsx to Power Logger Data Sreepur 123 (3).xlsx
Loaded: Power Logger Data Sreepur 123 (3).xlsx
Shape: (3499, 109)


,Time Stamp,UA,UB,UC,UAvg,UTHA,UTHB,UTHC,UTHAvg,IA,...,PDmIAVG_D/T,DmP,PDmP,PDmP_D/T,DmQ,PDmQ,PDmQ_D/T,DmS,PDmS,PDmS_D/T
0,2025-09-22 19:18:01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2025-07-08 20:10:59,0,172,2025-07-08 20:10:59,0,8,2025-07-08 20:09:59,0,452,2025-07-08 20:10:59
1,2025-09-22 19:19:01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2025-07-08 20:10:59,0,172,2025-07-08 20:10:59,0,8,2025-07-08 20:09:59,0,452,2025-07-08 20:10:59
2,2025-09-22 19:20:01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2025-07-08 20:10:59,0,172,2025-07-08 20:10:59,0,8,2025-07-08 20:09:59,0,452,2025-07-08 20:10:59
3,2025-09-22 19:21:01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2025-07-08 20:10:59,0,172,2025-07-08 20:10:59,0,8,2025-07-08 20:09:59,0,452,2025-07-08 20:10:59
4,2025-09-22 19:22:01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2025-07-08 20:10:59,0,172,2025-07-08 20:10:59,0,8,2025-07-08 20:09:59,0,452,2025-07-08 20:10:59



FINAL COMPOSITE STABILITY SUMMARY


/tmp/ipykernel_21811/3006089760.py:618: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["NP"] = cost_norm(df["Active_Power"], df["Active_Power"].max())
/tmp/ipykernel_21811/3006089760.py:619: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["NQ"] = cost_norm(df["Reactive_Power"], df["Reactive_Power"].max())
/tmp/ipykernel_21811/3006089760.py:620: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once 

,count,mean,std,min,25%,50%,75%,max
Voltage_Composite,3499.0,0.8623,0.0430,0.4250,0.8430,0.8653,0.8870,0.9466
Current_Composite,3499.0,0.4839,0.1496,0.2836,0.4217,0.4454,0.4677,1.0000
Frequency_Composite,3499.0,0.8989,0.0985,0.0700,0.8880,0.9177,0.9470,0.9998
Power_Factor_Composite,3499.0,0.8297,0.1338,0.1888,0.8590,0.8739,0.8849,0.9251
Load_Composite,3499.0,0.9710,0.0180,0.0983,0.9667,0.9695,0.9728,1.0000
Energy_Composite,3499.0,0.5351,0.0549,0.4322,0.4896,0.5401,0.5777,0.6775
Final_Composite_Stability_Index,3499.0,0.7672,0.0253,0.4738,0.7596,0.7694,0.7787,0.8454
Final_Composite_Stability_Percent,3499.0,76.7211,2.5337,47.3797,75.9599,76.9412,77.8666,84.5382



Saved: Full_Normalized_Final_Composite_Stability.xlsx
Stability value saved successfully.


,Feeder Name,Stability_Percent
0,Feeder 1,76.72


# **Composite Indicator Score**

In [ ]:
# ============================================================
# FINAL COMPOSITE INDICATOR CODE
# Uses values saved from four separate codes
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 1. Merge Four Indicator Values
# ============================================================

final_composite_df = reliability_value.merge(
    availability_value,
    on="Feeder Name",
    how="outer"
)

final_composite_df = final_composite_df.merge(
    customer_experience_value,
    on="Feeder Name",
    how="outer"
)

final_composite_df = final_composite_df.merge(
    stability_value,
    on="Feeder Name",
    how="left"
)

# If only Feeder 1 stability exists, apply same stability to other feeders
if "Stability_Percent" in final_composite_df.columns:
    final_composite_df["Stability_Percent"] = final_composite_df[
        "Stability_Percent"
    ].fillna(stability_value["Stability_Percent"].iloc[0])


# ============================================================
# 2. Ensure Percentage Format
# ============================================================

percent_cols = [
    "Reliability_Percent",
    "Availability_Percent",
    "Customer_Experience_Percent",
    "Stability_Percent"
]

for col in percent_cols:
    final_composite_df[col] = (
        pd.to_numeric(final_composite_df[col], errors="coerce")
        .clip(lower=0, upper=100)
    )


# ============================================================
# 3. Composite Indicator Weights
# ============================================================

COMPOSITE_WEIGHTS = {
    "Reliability_Percent": 0.25,
    "Availability_Percent": 0.25,
    "Customer_Experience_Percent": 0.25,
    "Stability_Percent": 0.25
}


def weighted_composite(row, weights):
    score = 0
    total_weight = 0

    for col, w in weights.items():
        if col in row.index and pd.notna(row[col]):
            score += row[col] * w
            total_weight += w

    if total_weight == 0:
        return np.nan

    return score / total_weight


# ============================================================
# 4. Final Composite Indicator Score
# ============================================================

final_composite_df["Composite_Indicator_Percent"] = final_composite_df.apply(
    lambda row: weighted_composite(row, COMPOSITE_WEIGHTS),
    axis=1
)


# ============================================================
# 5. Classification
# ============================================================

def classify_composite_indicator(score):
    if pd.isna(score):
        return "Insufficient Data"
    elif score >= 90:
        return "Excellent Composite Performance"
    elif score >= 75:
        return "Good Composite Performance"
    elif score >= 60:
        return "Moderate Composite Performance"
    elif score >= 40:
        return "Weak Composite Performance"
    else:
        return "Critical Composite Performance"


final_composite_df["Composite_Indicator_Class"] = final_composite_df[
    "Composite_Indicator_Percent"
].apply(classify_composite_indicator)


# ============================================================
# 6. Final Output
# ============================================================

final_cols = [
    "Feeder Name",
    "Reliability_Percent",
    "Availability_Percent",
    "Customer_Experience_Percent",
    "Stability_Percent",
    "Composite_Indicator_Percent",
    "Composite_Indicator_Class"
]

final_composite_df = final_composite_df[final_cols]

final_composite_df = final_composite_df.sort_values(
    "Composite_Indicator_Percent",
    ascending=False
)

print("\nFINAL FEEDER-WISE COMPOSITE INDICATOR SCORE")
display(final_composite_df.round(2))


# ============================================================
# 7. Export Result
# ============================================================

final_composite_df.to_excel(
    "final_feeder_wise_composite_indicator.xlsx",
    index=False
)

final_composite_df.to_csv(
    "final_feeder_wise_composite_indicator.csv",
    index=False
)

print("\nFinal composite indicator calculation completed successfully.")
print("Saved: final_feeder_wise_composite_indicator.xlsx")
print("Saved: final_feeder_wise_composite_indicator.csv")


FINAL FEEDER-WISE COMPOSITE INDICATOR SCORE


,Feeder Name,Reliability_Percent,Availability_Percent,Customer_Experience_Percent,Stability_Percent,Composite_Indicator_Percent,Composite_Indicator_Class
521,Feeder-7,47.85,87.60,43.54,76.72,63.93,Moderate Composite Performance
517,Feeder-7,47.85,87.52,43.54,76.72,63.91,Moderate Composite Performance
522,Feeder-7,47.85,87.60,43.40,76.72,63.89,Moderate Composite Performance
537,Feeder-7,47.64,87.60,43.54,76.72,63.88,Moderate Composite Performance
518,Feeder-7,47.85,87.52,43.40,76.72,63.87,Moderate Composite Performance
...,...,...,...,...,...,...,...
184,Feeder-11,44.95,86.63,40.61,76.72,62.23,Moderate Composite Performance
127,Feeder-10,44.53,86.87,40.75,76.72,62.22,Moderate Composite Performance
308,Feeder-3,44.34,86.51,41.19,76.72,62.19,Moderate Composite Performance
123,Feeder-10,44.53,86.61,40.75,76.72,62.15,Moderate Composite Performance



Final composite indicator calculation completed successfully.
Saved: final_feeder_wise_composite_indicator.xlsx
Saved: final_feeder_wise_composite_indicator.csv
